# 文本表达识别流程（情感倾向 + 压力表达倾向）

> **说明**：本流程仅用于文本研究，不构成诊断。所有输出只是对文本中**情感倾向**与**压力表达倾向**的估计，
> 措辞统一使用“倾向”“表达”“可能相关”。
>
> - en："This output is an estimate of textual tendency only and does not constitute a medical or psychological diagnosis."
> - zh："本结果仅为文本倾向估计，不构成医学或心理诊断。"

## README

**运行环境**：Google Colab Pro，优先 A100 / L4，调试可用 T4。所有输入输出位于 Google Drive 的 `DRIVE_ROOT`（默认 `/content/drive/MyDrive/MyProject`）。

### 执行顺序

首次运行：从上到下依次运行第 1 → 9 节（“运行时 → 全部运行”即可）。

再次“全部运行”是安全的：已有**完整训练**的 `models/{LANG}/model.pt` 时默认跳过训练（`SKIP_TRAINING_IF_MODEL_EXISTS = True`）。跳过的条件是：`model.pt` 与 `train_config.json` 都存在，`training_complete` 为 true，且记录的该语言训练 / 验证划分哈希与当前 `splits/` 一致；否则会打印原因（缺少配置 / 未完成训练 / 划分已变化）并重新训练。该语言测试已评估过时，阶段3 拒绝重写 `splits/`，阶段5 拒绝重新校准，阶段6（含零样本基线）拒绝重新运行（除非 `FORCE_RETEST = True`）。

### 断点恢复（Colab 断开 / 重启运行时之后）

后面阶段的“运行”单元会调用前面阶段“函数定义”单元里定义的函数，所以断开后**不能**只运行目标阶段的单元。单元编号指本 notebook 中从 0 开始数的位置（本 README 为单元 0），括号内为单元首行注释。函数定义单元不写任何文件，重复运行是安全的。

1. 每次断开后都要先运行：单元 2（安装依赖）、3（环境检查）、4（挂载 Drive）、6（配置区）、7（默认标签映射）、8（通用工具与目录检查）。
2. 再运行目标阶段所需的函数定义单元，最后运行该阶段的“运行”单元：

| 要运行的阶段（运行单元） | 需先重新运行的函数定义单元 |
|---|---|
| 阶段2 标签映射（单元 12） | 10（阶段2：映射表读取、校验与标签规则）、11（阶段2：逐数据源映射、合并、分布报告与文档） |
| 阶段3 划分与泄露检查（单元 16） | 11、14（阶段3：去重、分层与划分）、15（阶段3：泄露检查） |
| 阶段4 训练（单元 21） | 11、18（阶段4：模型定义）、19（阶段4：数据加载、掩码损失与评估指标）、20（阶段4：训练循环、早停、模型保存与实验日志） |
| 阶段5 校准与阈值（单元 25） | 11、18、19、20、23（阶段5：校准结果读取与模型指纹一致性检查）、24（阶段5：温度缩放与阈值搜索） |
| 阶段6 测试评估（单元 28，然后单元 29 零样本基线） | 11、18、19、20、23、27（阶段6：评估与误差分析函数） |
| 阶段7 推理接口（单元 31 定义接口，单元 32 示例） | 18、19、20、23 |
| 第 9 节 输出文件列表（单元 34） | 无（只需第 1 步） |
| 附 表述自检（单元 36） | 无（只需第 1 步） |

最省事的做法：完成第 1 步后，依次运行全部函数定义单元 10、11、14、15、18、19、20、23、24、27，再运行目标阶段的运行单元。

| 节 | 阶段 | 输入 | 输出 |
|---|---|---|---|
| 1 | 环境检查与 Drive 挂载 | — | 打印 GPU / 库版本；挂载 `/content/drive` |
| 2 | 配置区 | — | 检查并创建目录；若缺失则写入 `configs/label_mapping.json`（默认示例） |
| 3 | 阶段2：标签映射 | `raw/*`（只读）、`configs/label_mapping.json` | `processed/unified.csv`、`reports/mapping_distribution.csv`、`docs/label_mapping_notes.md`、`docs/data_inventory.csv` |
| 4 | 阶段3：划分与泄露检查 | `processed/unified.csv` | `splits/train.csv`、`splits/val.csv`、`splits/test.csv`、`reports/leakage_check.txt` |
| 5 | 阶段4：训练 | `splits/train.csv`、`splits/val.csv` | `models/{LANG}/model.pt`、`models/{LANG}/tokenizer/`、`models/{LANG}/encoder_config/`、`models/{LANG}/train_config.json`；追加 `reports/experiment_log.csv` |
| 6 | 阶段5：校准与阈值 | `splits/val.csv`、`models/{LANG}/` | `configs/temperature.json`、`configs/thresholds.json` |
| 7 | 阶段6：测试评估与误差分析（只运行一次） | `splits/test.csv`、`models/{LANG}/`、`configs/*.json` | `reports/test_results.json`、`reports/error_cases.csv`、`reports/baseline_results.json`、标记文件 `reports/.test_evaluated_{LANG}` |
| 8 | 阶段7：推理接口与示例 | `models/{LANG}/`、`configs/*.json` | `analyze()` / `analyze_batch()`；追加 `reports/inference_log.jsonl` |
| 9 | 输出文件列表 | — | 按阶段打印已存在的文件及大小 |
| 附 | 表述自检 | 本 notebook 源码 | 断言不出现禁用表述（`BANNED_WORDING`） |

### 关键约定

- **切换语言**：只需修改配置区的 `LANG`（`"en"` / `"zh"`）。模型保存在 `models/{LANG}/`，`temperature.json`、`thresholds.json`、`test_results.json`、`baseline_results.json` 均按语言分键保存，两种语言互不覆盖。
- **统一列**：`text, lang, source, sentiment(-2..2), stress(0/1), anxiety(0/1), depression(0/1), label_method(direct/mapped)`，另附辅助列 `official_split`（官方划分，供阶段3使用）。不能映射的字段保持为空值（NaN），绝不填 0。`anxiety` / `depression` 只作为“可能相关的表达”标签被映射保存，当前模型只训练情感与压力两个头。
- **label_method**：在 `label_mapping.json` 中按数据源、按维度声明。人工标注（含对人工评分做确定性变换，如 EmoBank valence 分箱）为 `direct`；凡由代理信号（子版块名、关键词、LLM 输出等）推得的标签必须为 `mapped`。一行中任一非空标签来自 `mapped` 规则，则整行为 `mapped`。
- **EmoBank 取整**：按规范字面计算 `x = (valence − 3) × 2 / 2`，然后**四舍五入（远离零）**：`sign(x) · floor(|x| + 0.5)`，再限制在 [−2, 2]。不使用 Python 内置 `round`，因为它是“银行家舍入”（`round(0.5)=0`、`round(1.5)=2`、`round(2.5)=2`），对 ±0.5 边界的处理随数值奇偶而变，不对称；远离零取整对正负对称（±0.5 → ±1），且在不同 numpy / pandas 版本下结果可复现。
- **数据划分**：先删除无任何标签的行，再按规范化文本（去首尾空格、统一大小写、合并内部空白）去重（优先保留 direct 行）。每种语言独立划分（随机种子 42），因此加入另一种语言的数据不会改变已有语言的划分。direct 行按 `lang|source|标签组合` 分层 70/15/15；样本数 < 3 的层合并为该数据源的 `rare` 层，仍不足则并入全局 `rare` 层。Dreaddit 沿用官方划分：官方 test → test，官方 train → 85/15 划为 train/val。mapped 行只进入 train/val（85/15）。代码断言测试集全部为 `direct`。
- **泄露检查**：任一规范化文本出现在多个划分中即写出报告并报错。
- **模型**：共享编码器 + 情感有序头（累积链接 / 比例优势模型：一个标量分数 + 4 个有序切点，P(y ≤ k) = sigmoid((c_k − s)/T)）+ 压力头（1 个 logit，BCE）。两个损失都带掩码：缺失某维度标签的样本不计入该损失。
- **学习率**：编码器使用 `LEARNING_RATE = 1e-5`；两个输出头（含有序切点）使用 `HEAD_LEARNING_RATE = 1e-3`（独立的 AdamW 参数组，切点与偏置不做权重衰减），否则切点几乎不离开初始值，各等级宽度学不出来。两个学习率都记录在 `experiment_log.csv`。
- **显存与 batch**：roberta-large 建议在 A100 / L4 上使用。显存较小的 GPU（如 T4）可把配置区 `BATCH_SIZE` 改为 8，梯度累积步数会自动变为 4，等效 batch 仍为 32。混合精度只在 GPU 原生支持 bf16（计算能力 ≥ 8，如 A100 / L4）时用 bf16，否则用 fp16 + GradScaler；两个输出头始终以 fp32 计算。
- **训练前先看分布**：训练前请先检查 `reports/mapping_distribution.csv`。EmoBank 的 valence 集中在 3 附近，大多数行会映射为情感等级 0，类别明显不均衡，需要据此判断是否补充数据或调整采样。
- **早停**：验证集指标 = 0.5 × 情感 QWK + 0.5 × 压力 F1（训练中压力阈值固定 0.5，因为校准阈值尚未产生），patience = 2。
- **校准**：温度缩放与压力阈值搜索只使用验证集；情感输出不使用阈值。
- **测试只运行一次**：阶段6 的测试评估与零样本基线都完成后，才写入标记文件 `reports/.test_evaluated_{LANG}`（内含模型指纹）；之后阶段6 拒绝再次运行，阶段3 也拒绝重写 `splits/`，除非配置区 `FORCE_RETEST = True`。
- **模型指纹**：阶段5 把 `models/{LANG}/model.pt` 的 SHA-256 写入 `temperature.json` / `thresholds.json` 的该语言条目。阶段6、7 加载模型时核对指纹，不一致（例如重新训练过）会报错并提示重新运行阶段5。
- **help_message**：只在输入包含配置区列出的明确自伤 / 自杀相关关键词时出现（大小写不敏感的子串匹配；无论 `LANG` 为何，中英文两个关键词表都会检查，求助文本使用 `LANG` 对应的语言）。**这只是粗略的提示，不是风险评估**：会漏掉没有使用这些关键词的表达，也会误触发（例如引用、否定句）。
- **推理日志**：`reports/inference_log.jsonl` 只记录输入文本的 SHA-256 哈希与输出，不保存原文。
- **中文数据**：用户尚未指定中文数据集。`label_mapping.json` 中的 `zh_example` 是占位示例，请替换为实际数据源与列名后再以 `LANG = "zh"` 运行。
- **SMOKE_TEST**：仅用于本地 CPU 冒烟测试（极小模型、1 个 epoch），正式运行保持 `False`。

## 1. 环境检查与 Drive 挂载

In [ ]:
# 安装固定版本依赖（Colab 已自带 torch）
!pip install -q transformers==4.44.2 datasets==2.21.0 scikit-learn==1.5.1 accelerate==0.33.0 pandas

In [ ]:
# 环境检查：Python / 库版本 / GPU 型号与精度支持
import sys
import platform

import numpy as np
import pandas as pd
import sklearn
import torch
import transformers


BF16_MIN_COMPUTE_CAPABILITY = 8  # 原生 bf16 需要计算能力 ≥ 8（Ampere 及以上，如 A100 / L4）


def bf16_native_supported():
    """仅当 GPU 原生支持 bf16 时返回 True（不计入模拟支持；T4 等计算能力 < 8 的 GPU 返回 False）。"""
    if not torch.cuda.is_available():
        return False
    if torch.cuda.get_device_capability(0)[0] < BF16_MIN_COMPUTE_CAPABILITY:
        return False
    try:
        return bool(torch.cuda.is_bf16_supported(including_emulation=False))
    except TypeError:  # 旧版 torch 没有 including_emulation 参数
        return bool(torch.cuda.is_bf16_supported())


def describe_environment():
    """打印运行环境信息（Python、关键库版本、GPU 型号、原生 bf16 支持），并返回 GPU 名称字符串。"""
    print(f"Python        : {sys.version.split()[0]} ({platform.platform()})")
    print(f"torch         : {torch.__version__}")
    print(f"transformers  : {transformers.__version__}")
    print(f"scikit-learn  : {sklearn.__version__}")
    print(f"pandas        : {pd.__version__}")
    print(f"numpy         : {np.__version__}")
    if torch.cuda.is_available():
        name = torch.cuda.get_device_name(0)
        mem_gb = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3
        print(f"GPU           : {name} ({mem_gb:.1f} GB), 原生 bf16 支持: {bf16_native_supported()}")
        # 推荐 A100 / L4；T4 仅建议调试使用
        if not any(tag in name for tag in ("A100", "L4", "H100")):
            print("提示：当前不是 A100 / L4，建议仅用于调试（T4 不支持 bf16，将使用 fp16）。")
        return name
    print("GPU           : 未检测到 CUDA，将使用 CPU（仅适合冒烟测试）")
    return "CPU"


GPU_NAME = describe_environment()

In [ ]:
# 挂载 Google Drive（所有输入输出都在 Drive 指定目录中）
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("当前不在 Colab 环境中，跳过 Drive 挂载。")

## 2. 配置区

所有可变项（语言、编码器、路径、超参数、阈值搜索范围、置信度分档、关键词、免责声明等）都集中在下面的单元中；函数内部不硬编码这些值。

In [ ]:
# ==================== 配置区：所有可变项集中于此 ====================
import os

# ---- 1. 语言：修改此处即可切换语言，其余代码无需改动 ----
LANG = "en"  # "en" 或 "zh"
SUPPORTED_LANGS = ("en", "zh")
assert LANG in SUPPORTED_LANGS, f"LANG 必须是 {SUPPORTED_LANGS} 之一"  # 必须在下面按语言查路径之前检查

# ---- 冒烟测试开关：仅用于本地 CPU 验证流程，正式运行务必保持 False ----
SMOKE_TEST = False

# ---- 2. 编码器映射 ----
# en: "base" = roberta-base（默认，调试用）；"large" = roberta-large（正式训练，显存足够时）
# zh: 只有 hfl/chinese-roberta-wwm-ext，ENCODER_VARIANT 对中文不起作用
ENCODER_VARIANT = "base"
ENCODERS = {
    "en": {"base": "roberta-base", "large": "roberta-large"},
    "zh": {"base": "hfl/chinese-roberta-wwm-ext", "large": "hfl/chinese-roberta-wwm-ext"},
}
LARGE_MODEL_MIN_GPU_MEM_GB = 20  # 使用 large 时若显存低于该值则给出警告

# ---- 3. 路径（全部由 DRIVE_ROOT 派生）----
DRIVE_ROOT = "/content/drive/MyDrive/MyProject"
DIRS = {
    "raw": os.path.join(DRIVE_ROOT, "raw"),              # 原始数据，只读
    "processed": os.path.join(DRIVE_ROOT, "processed"),  # 统一格式后的数据
    "splits": os.path.join(DRIVE_ROOT, "splits"),        # train/val/test
    "models_zh": os.path.join(DRIVE_ROOT, "models", "zh"),
    "models_en": os.path.join(DRIVE_ROOT, "models", "en"),
    "reports": os.path.join(DRIVE_ROOT, "reports"),      # 评估结果与误差分析
    "configs": os.path.join(DRIVE_ROOT, "configs"),      # 映射表、阈值、温度参数
    "docs": os.path.join(DRIVE_ROOT, "docs"),            # 映射说明与数据清单
}
MODEL_DIR = DIRS[f"models_{LANG}"]
TEST_MARKER_TEMPLATE = os.path.join(DIRS["reports"], ".test_evaluated_{lang}")  # 每种语言一个测试标记文件
FILES = {
    # configs/
    "label_mapping": os.path.join(DIRS["configs"], "label_mapping.json"),
    "temperature": os.path.join(DIRS["configs"], "temperature.json"),
    "thresholds": os.path.join(DIRS["configs"], "thresholds.json"),
    # processed/ 与 splits/
    "unified": os.path.join(DIRS["processed"], "unified.csv"),
    "split_train": os.path.join(DIRS["splits"], "train.csv"),
    "split_val": os.path.join(DIRS["splits"], "val.csv"),
    "split_test": os.path.join(DIRS["splits"], "test.csv"),
    # reports/
    "mapping_distribution": os.path.join(DIRS["reports"], "mapping_distribution.csv"),
    "leakage_check": os.path.join(DIRS["reports"], "leakage_check.txt"),
    "experiment_log": os.path.join(DIRS["reports"], "experiment_log.csv"),
    "test_results": os.path.join(DIRS["reports"], "test_results.json"),
    "error_cases": os.path.join(DIRS["reports"], "error_cases.csv"),
    "baseline_results": os.path.join(DIRS["reports"], "baseline_results.json"),
    "inference_log": os.path.join(DIRS["reports"], "inference_log.jsonl"),
    "test_marker": TEST_MARKER_TEMPLATE.format(lang=LANG),
    # docs/
    "mapping_notes": os.path.join(DIRS["docs"], "label_mapping_notes.md"),
    "data_inventory": os.path.join(DIRS["docs"], "data_inventory.csv"),
    # models/{LANG}/（不同语言互不覆盖）
    "model_weights": os.path.join(MODEL_DIR, "model.pt"),
    "train_config": os.path.join(MODEL_DIR, "train_config.json"),
    "tokenizer_dir": os.path.join(MODEL_DIR, "tokenizer"),
    "encoder_config_dir": os.path.join(MODEL_DIR, "encoder_config"),
}

# ---- 4. 超参数 ----
LEARNING_RATE = 1e-5          # 编码器学习率
HEAD_LEARNING_RATE = 1e-3     # 两个输出头（含有序切点）的学习率
BATCH_SIZE = 16
EFFECTIVE_BATCH_SIZE = 32                                   # 通过梯度累积达到等效 32
GRAD_ACCUM_STEPS = max(1, EFFECTIVE_BATCH_SIZE // BATCH_SIZE)  # = 2
MAX_EPOCHS = 6
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
MAX_LENGTH = 256
MAX_GRAD_NORM = 1.0
DROPOUT = 0.1
EVAL_BATCH_SIZE = 64
SKIP_TRAINING_IF_MODEL_EXISTS = True  # models/{LANG}/model.pt 已存在时跳过训练、加载已有模型；设为 False 才会重新训练

# ---- 5. 损失权重 ----
W_SENT = 1.0
W_STRESS = 1.0

# ---- 6. 随机种子 ----
SEED = 42

# ---- 7. 早停 ----
EARLY_STOP_PATIENCE = 2
EARLY_STOP_WEIGHTS = {"sentiment_qwk": 0.5, "stress_f1": 0.5}
EARLY_STOP_STRESS_THRESHOLD = 0.5  # 训练期间压力 F1 使用的固定阈值（校准阈值尚未产生）

# ---- 标签定义 ----
SENTIMENT_LABELS = [-2, -1, 0, 1, 2]
BINARY_LABEL_VALUES = [0, 1]
LABEL_DIMENSIONS = ["sentiment", "stress", "anxiety", "depression"]
UNIFIED_COLUMNS = ["text", "lang", "source", "sentiment", "stress", "anxiety", "depression", "label_method"]
OFFICIAL_SPLIT_COLUMN = "official_split"  # 辅助列：记录官方划分
SIGNATURE_ABBREVIATIONS = {"sentiment": "s", "stress": "st", "anxiety": "ax", "depression": "dp"}
ORDINAL_INIT_CUTPOINTS = [-1.5, -0.5, 0.5, 1.5]  # 有序头切点初始值（4 个，对应 5 类）
EMOBANK_VALENCE_CENTER = 3.0  # EmoBank valence 中点
NUMERIC_EPS = 1e-8

# ---- 数据划分 ----
SPLIT_RATIOS = {"train": 0.70, "val": 0.15, "test": 0.15}   # direct 数据
MAPPED_SPLIT_RATIOS = {"train": 0.85, "val": 0.15}          # mapped 数据只进 train/val
OFFICIAL_TRAIN_POOL_RATIOS = {"train": 0.85, "val": 0.15}   # 官方 train 再划分为 train/val
MIN_STRATUM_SIZE = 3
OFFICIAL_SPLIT_ALIASES = {"train": "train", "training": "train", "dev": "val", "val": "val",
                          "valid": "val", "validation": "val", "test": "test"}

# ---- 校准与阈值 ----
TEMPERATURE_BOUNDS = (0.05, 20.0)
TEMPERATURE_LBFGS_MAX_ITER = 100
TEMPERATURE_LBFGS_LR = 0.1
THRESHOLD_SEARCH = {"start": 0.05, "stop": 0.95, "step": 0.01}

# ---- 测试评估 ----
FORCE_RETEST = False  # True 时允许重新运行阶段6、允许重写已评估语言的 splits/（会覆盖该语言的测试结果）
N_ERROR_CASES = 30

# ---- 推理 ----
SENTIMENT_CONFIDENCE_BANDS = {"high": 0.70, "medium": 0.45}  # 按最大概率分档
STRESS_CONFIDENCE_BANDS = {"high": 0.30, "medium": 0.15}     # 按 |score − threshold| 分档
MIN_INPUT_UNITS = {"en": 5, "zh": 5}  # en：少于 5 个词；zh：少于 5 个字 → insufficient_info
CJK_CHAR_PATTERN = r"[㐀-䶿一-鿿豈-﫿]"
ALNUM_TOKEN_PATTERN = r"[A-Za-z0-9]+"
OUTPUT_DECIMALS = 4
INFERENCE_LOG_ENABLED = True

# ---- 8. 免责声明与求助文本（按语言分别提供）----
DISCLAIMERS = {
    "en": "This output is an estimate of textual tendency only and does not constitute a medical or psychological diagnosis.",
    "zh": "本结果仅为文本倾向估计，不构成医学或心理诊断。",
}
HELP_MESSAGES = {
    "en": "If you are going through something difficult, please contact local emergency services or a crisis line. In Australia, Lifeline is available on 13 11 14.",
    "zh": "如果你正在经历困难，请联系当地紧急服务或危机热线。澳大利亚可拨打 Lifeline 13 11 14。",
}
# 明确的自伤 / 自杀相关关键词（大小写不敏感子串匹配）；仅为粗略提示，不是风险评估
SELF_HARM_KEYWORDS = {
    "en": ["suicide", "suicidal", "kill myself", "end my life", "self-harm", "self harm",
           "cut myself", "want to die", "hurt myself"],
    "zh": ["自杀", "想死", "不想活", "结束生命", "自残", "割腕", "轻生"],
}

# ---- 零样本基线（预训练情感模型；label_values 按模型输出类别顺序映射到情感等级）----
BASELINE_MODELS = {
    "en": {"name": "cardiffnlp/twitter-roberta-base-sentiment-latest", "label_values": [-1, 0, 1]},
    "zh": {"name": "IDEA-CCNL/Erlangshen-Roberta-110M-Sentiment", "label_values": [-1, 1]},
}
BASELINE_STRESS_NOTE = "no pretrained stress baseline"

# ---- 表述自检（附录单元使用）----
SELF_CHECK_NOTEBOOK_PATH = None  # 非 Colab 环境下可指定 .ipynb 路径用于自检
WORDING_ALLOW_MARKER = "wording-check: allow"
WORDING_WHITELIST = list(DISCLAIMERS.values()) + ["不构成诊断"]
BANNED_WORDING = ["诊断", "确诊", "患有", "抑郁症", "diagnos", "suffers from", "suffering from"]  # wording-check: allow

# ---- 冒烟测试专用设置（SMOKE_TEST = True 时生效）----
SMOKE_ENCODER_NAME = "hf-internal-testing/tiny-random-roberta"
SMOKE_BASELINE_MODELS = {
    "en": {"name": "hf-internal-testing/tiny-random-RobertaForSequenceClassification", "label_values": [-1, 1]},
    "zh": {"name": "hf-internal-testing/tiny-random-RobertaForSequenceClassification", "label_values": [-1, 1]},
}
SMOKE_SCRATCH_ENCODER = {"hidden_size": 32, "num_hidden_layers": 2, "num_attention_heads": 2,
                         "intermediate_size": 64}
SMOKE_MAX_EPOCHS = 1
SMOKE_MAX_LENGTH = 64

# ---- 派生设置 ----
if SMOKE_TEST:
    ENCODER_NAME = SMOKE_ENCODER_NAME
    MAX_EPOCHS = SMOKE_MAX_EPOCHS
    MAX_LENGTH = SMOKE_MAX_LENGTH
    BASELINE_MODELS = SMOKE_BASELINE_MODELS
    print("⚠ SMOKE_TEST = True：使用极小模型，仅用于流程验证。")
else:
    ENCODER_NAME = ENCODERS[LANG][ENCODER_VARIANT]

assert abs(sum(SPLIT_RATIOS.values()) - 1.0) < 1e-9
print(f"LANG = {LANG} | 编码器 = {ENCODER_NAME} | DRIVE_ROOT = {DRIVE_ROOT}")
print(f"batch = {BATCH_SIZE} × 累积 {GRAD_ACCUM_STEPS} = 等效 {BATCH_SIZE * GRAD_ACCUM_STEPS}")

In [ ]:
# 默认标签映射：仅当 configs/label_mapping.json 不存在时写入（路径相对 DRIVE_ROOT）
DEFAULT_LABEL_MAPPING = {
    "_comment": (
        "Label mapping for stage 2. Paths are relative to DRIVE_ROOT and must live under raw/. "
        "rule: emobank_valence (sentiment only) | identity | value_map (requires value_map). "
        "method: direct = human annotation (incl. deterministic transform of human ratings); "
        "mapped = derived from a proxy (subreddit, keyword, LLM output...). Missing label keys -> NaN."
    ),
    "sources": {
        "emobank": {
            "lang": "en",
            "files": ["raw/emobank/emobank.csv"],
            "format": "csv",
            "text_column": "text",
            "labels": {
                "sentiment": {"column": "V", "rule": "emobank_valence", "method": "direct"}
            },
            "official_split_column": None,
            "official_split_files": None,
            "notes": "EmoBank columns: id, split, V, A, D, text. Its 'split' column is NOT treated as an official split for this task.",
        },
        "dreaddit": {
            "lang": "en",
            "files": ["raw/dreaddit/dreaddit-train.csv", "raw/dreaddit/dreaddit-test.csv"],
            "format": "csv",
            "text_column": "text",
            "labels": {
                "stress": {"column": "label", "rule": "identity", "method": "direct"}
            },
            "official_split_column": None,
            "official_split_files": {
                "train": ["raw/dreaddit/dreaddit-train.csv"],
                "test": ["raw/dreaddit/dreaddit-test.csv"],
            },
            "notes": "Dreaddit human stress annotation (0/1). Official train/test split is kept.",
        },
        "zh_example": {
            "lang": "zh",
            "files": ["raw/zh_example/zh_data.csv"],
            "format": "csv",
            "text_column": "text",
            "labels": {
                "sentiment": {"column": "sentiment", "rule": "identity", "method": "direct"},
                "stress": {"column": "stress", "rule": "value_map",
                           "value_map": {"1": 1, "0": 0, "是": 1, "否": 0}, "method": "direct"},
            },
            "official_split_column": None,
            "official_split_files": None,
            "notes": "FILL ME IN: placeholder only - no Chinese dataset has been specified. Replace files/columns/rules with a real source; use method 'mapped' for any proxy- or LLM-derived label.",
        },
    },
}

In [ ]:
# 通用工具与目录检查
import datetime
import hashlib
import json
import math
import random
import re
import time
import warnings


def set_seed(seed):
    """设置 Python、NumPy、PyTorch（含 CUDA）的随机种子，并开启 cuDNN 确定性模式。"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def ensure_directories(dirs):
    """检查约定目录是否存在，不存在则创建；返回新建目录列表。"""
    created = []
    for key, path in dirs.items():
        if not os.path.isdir(path):
            os.makedirs(path, exist_ok=True)
            created.append(path)
        print(f"  [{'新建' if path in created else '已存在'}] {key:10s} {path}")
    return created


def assert_not_raw(path):
    """确保写入路径不在 raw/ 目录下（raw 为只读原始数据）。"""
    raw_root = os.path.realpath(DIRS["raw"])
    target = os.path.realpath(path)
    if target == raw_root or target.startswith(raw_root + os.sep):
        raise PermissionError(f"禁止写入 raw/ 目录：{path}")


def to_jsonable(obj):
    """把 numpy 类型、NaN 等转换为可写入标准 JSON 的 Python 对象（NaN → None）。"""
    if isinstance(obj, dict):
        return {str(k): to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_jsonable(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return to_jsonable(obj.tolist())
    if isinstance(obj, (np.bool_, bool)):
        return bool(obj)
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating, float)):
        value = float(obj)
        return None if (math.isnan(value) or math.isinf(value)) else value
    return obj


def save_json(obj, path):
    """以 UTF-8、缩进格式保存 JSON（自动转换 NaN 等），禁止写入 raw/。"""
    assert_not_raw(path)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(to_jsonable(obj), f, ensure_ascii=False, indent=2)


def load_json(path, default=None):
    """读取 JSON 文件；文件不存在时返回 default。"""
    if not os.path.exists(path):
        return default
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def update_lang_entry(path, lang, entry):
    """在按语言分键的 JSON 文件中只更新 lang 对应的条目，保留其他语言的结果。"""
    data = load_json(path, default={}) or {}
    data[lang] = entry
    save_json(data, path)
    return data


def save_csv(df, path, **kwargs):
    """保存 CSV（UTF-8），禁止写入 raw/。"""
    assert_not_raw(path)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    df.to_csv(path, index=False, encoding="utf-8", **kwargs)


def utc_now_iso():
    """返回当前 UTC 时间的 ISO 8601 字符串。"""
    return datetime.datetime.now(datetime.timezone.utc).isoformat()


def file_sha256(path):
    """计算文件的 SHA-256（只读），用于数据清单与模型指纹。"""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def model_fingerprint():
    """返回当前语言 models/{LANG}/model.pt 的 SHA-256 指纹；模型不存在时返回 None。"""
    path = FILES["model_weights"]
    return file_sha256(path) if os.path.exists(path) else None


def write_default_label_mapping(path, mapping):
    """若 configs/label_mapping.json 不存在，则写入默认示例映射；已存在时不覆盖。"""
    if os.path.exists(path):
        print(f"  映射表已存在，不覆盖：{path}")
        return False
    save_json(mapping, path)
    print(f"  已写入默认映射表（请按实际数据检查）：{path}")
    return True


# 关键步骤：开始时检查目录，不存在则创建
print("目录检查：")
ensure_directories(DIRS)
write_default_label_mapping(FILES["label_mapping"], DEFAULT_LABEL_MAPPING)
set_seed(SEED)

## 3. 阶段2：标签映射

- **输入**：`raw/` 下的原始文件（只读）、`configs/label_mapping.json`
- **输出**：`processed/unified.csv`、`reports/mapping_distribution.csv`、`docs/label_mapping_notes.md`、`docs/data_inventory.csv`
- 不能映射的字段保留为 NaN，不填 0。映射表中缺失文件的数据源会被跳过并给出警告（例如尚未填写的 `zh_example`）。

In [ ]:
# 阶段2：映射表读取、校验与标签规则
VALID_RULES = ("emobank_valence", "identity", "value_map")
VALID_METHODS = ("direct", "mapped")
VALID_FORMATS = ("csv", "tsv", "jsonl")


def load_label_mapping(path):
    """读取 configs/label_mapping.json 并校验，返回映射配置字典。"""
    mapping = load_json(path)
    if mapping is None:
        raise FileNotFoundError(f"找不到映射表：{path}")
    validate_label_mapping(mapping)
    return mapping


def iter_sources(mapping):
    """遍历映射表中的数据源（跳过以下划线开头的注释键）。"""
    for name, cfg in mapping["sources"].items():
        if not name.startswith("_"):
            yield name, cfg


def validate_label_mapping(mapping):
    """校验映射表结构；发现问题时一次性列出全部错误并抛出 ValueError。"""
    errors = []
    if not isinstance(mapping, dict) or not isinstance(mapping.get("sources"), dict) or not mapping["sources"]:
        raise ValueError("label_mapping.json 必须包含非空的 'sources' 字典")
    for name, cfg in iter_sources(mapping):
        prefix = f"[{name}]"
        if cfg.get("lang") not in SUPPORTED_LANGS:
            errors.append(f"{prefix} lang 必须是 {SUPPORTED_LANGS} 之一")
        files = cfg.get("files")
        if not isinstance(files, list) or not files:
            errors.append(f"{prefix} files 必须是非空列表")
            files = []
        for f in files:
            if not (isinstance(f, str) and f.replace("\\", "/").startswith("raw/")):
                errors.append(f"{prefix} 文件路径必须位于 raw/ 下：{f}")
        if cfg.get("format") not in VALID_FORMATS:
            errors.append(f"{prefix} format 必须是 {VALID_FORMATS} 之一")
        if not isinstance(cfg.get("text_column"), str):
            errors.append(f"{prefix} 缺少 text_column")
        if "labels" not in cfg:
            errors.append(f"{prefix} 缺少 labels 键（没有可映射的标签时也请写空字典 {{}}）")
            labels = {}
        elif not isinstance(cfg["labels"], dict):
            errors.append(f"{prefix} labels 必须是字典")
            labels = {}
        else:
            labels = cfg["labels"]
        for dim, rule_cfg in labels.items():
            if dim not in LABEL_DIMENSIONS:
                errors.append(f"{prefix} 未知标签维度：{dim}")
                continue
            if not isinstance(rule_cfg.get("column"), str):
                errors.append(f"{prefix}.{dim} 缺少 column")
            rule = rule_cfg.get("rule")
            if rule not in VALID_RULES:
                errors.append(f"{prefix}.{dim} rule 必须是 {VALID_RULES} 之一")
            if rule == "emobank_valence" and dim != "sentiment":
                errors.append(f"{prefix}.{dim} emobank_valence 规则只能用于 sentiment")
            if rule == "value_map" and not isinstance(rule_cfg.get("value_map"), dict):
                errors.append(f"{prefix}.{dim} value_map 规则需要 value_map 字典")
            if rule_cfg.get("method") not in VALID_METHODS:
                errors.append(f"{prefix}.{dim} method 必须是 {VALID_METHODS} 之一")
        split_col = cfg.get("official_split_column")
        split_files = cfg.get("official_split_files")
        if split_col is not None and split_files is not None:
            errors.append(f"{prefix} official_split_column 与 official_split_files 只能二选一")
        if split_col is not None and not isinstance(split_col, str):
            errors.append(f"{prefix} official_split_column 必须是字符串或 null")
        if split_files is not None:
            if not isinstance(split_files, dict):
                errors.append(f"{prefix} official_split_files 必须是字典或 null")
            else:
                for split_name, split_list in split_files.items():
                    if split_name not in OFFICIAL_SPLIT_ALIASES:
                        errors.append(f"{prefix} 未知官方划分名：{split_name}")
                    for f in split_list:
                        if f not in files:
                            errors.append(f"{prefix} official_split_files 中的 {f} 不在 files 列表中")
    if errors:
        raise ValueError("label_mapping.json 校验失败：\n  " + "\n  ".join(errors))
    return True


def read_raw_file(rel_path, fmt):
    """只读方式读取 raw/ 下的一个原始文件，所有列按字符串读入（不修改原文件）。"""
    full_path = os.path.join(DRIVE_ROOT, rel_path)
    if fmt == "csv":
        return pd.read_csv(full_path, dtype=str)
    if fmt == "tsv":
        return pd.read_csv(full_path, sep="\t", dtype=str)
    if fmt == "jsonl":
        return pd.read_json(full_path, lines=True, dtype=False)
    raise ValueError(f"不支持的格式：{fmt}")


def emobank_valence_to_sentiment(valence):
    """EmoBank valence（1–5）→ sentiment（−2..2）：字面计算 (v−3)×2/2，远离零四舍五入，再裁剪。"""
    v = pd.to_numeric(valence, errors="coerce").astype(float)
    x = (v - EMOBANK_VALENCE_CENTER) * 2 / 2           # 按规范字面实现
    rounded = np.sign(x) * np.floor(np.abs(x) + 0.5)   # 远离零四舍五入（不用银行家舍入）
    return rounded.clip(min(SENTIMENT_LABELS), max(SENTIMENT_LABELS))  # NaN 保持为 NaN


def _normalize_map_key(value):
    """把原始值规范化为 value_map 的字符串键（例如 1.0 或 "1.0" → "1"，去除首尾空格）。"""
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return None
    if isinstance(value, (int, np.integer)):
        return str(int(value))
    if isinstance(value, (float, np.floating)) and float(value).is_integer():
        return str(int(value))
    text = str(value).strip()
    try:
        number = float(text)
        if number.is_integer():  # 数字字符串如 "1.0" → "1"
            return str(int(number))
    except ValueError:
        pass
    return text


def apply_label_rule(raw_series, dim, rule_cfg, source_name):
    """按规则把原始列转换为统一标签；无法映射或取值非法的保留为 NaN（不填 0）。"""
    rule = rule_cfg["rule"]
    if rule == "emobank_valence":
        values = emobank_valence_to_sentiment(raw_series)
    elif rule == "identity":
        values = pd.to_numeric(raw_series, errors="coerce").astype(float)
    else:  # value_map
        vmap = {_normalize_map_key(k): v for k, v in rule_cfg["value_map"].items()}  # 键与原始值用同一规范化
        values = raw_series.map(lambda v: vmap.get(_normalize_map_key(v), np.nan))
        values = pd.to_numeric(values, errors="coerce").astype(float)
    # 关键步骤：只保留合法取值，其余设为 NaN
    allowed = SENTIMENT_LABELS if dim == "sentiment" else BINARY_LABEL_VALUES
    invalid = values.notna() & ~values.isin(allowed)
    if invalid.any():
        print(f"  ⚠ [{source_name}.{dim}] {int(invalid.sum())} 个取值不在 {allowed} 内，已置为 NaN")
        values = values.mask(invalid)
    unmapped = raw_series.notna() & values.isna()
    if unmapped.any():
        print(f"  ⚠ [{source_name}.{dim}] {int(unmapped.sum())} 个非空原始值无法映射，保留为 NaN")
    return values

In [ ]:
# 阶段2：逐数据源映射、合并、分布报告与文档
def map_source(name, cfg):
    """把一个数据源的全部原始文件映射为统一列；返回 (DataFrame 或 None, 数据清单记录列表)。"""
    frames, inventory = [], []
    split_by_file = {}
    for split_name, split_files in (cfg.get("official_split_files") or {}).items():
        for f in split_files:
            split_by_file[f] = OFFICIAL_SPLIT_ALIASES[split_name]
    for rel_path in cfg["files"]:
        full_path = os.path.join(DRIVE_ROOT, rel_path)
        record = {"source": name, "lang": cfg["lang"], "file": rel_path, "exists": os.path.exists(full_path),
                  "rows": None, "sha256": None}
        inventory.append(record)
        if not record["exists"]:
            print(f"  ⚠ [{name}] 文件不存在，跳过：{rel_path}")
            continue
        raw = read_raw_file(rel_path, cfg["format"])
        record["rows"], record["sha256"] = len(raw), file_sha256(full_path)
        needed = [cfg["text_column"]] + [r["column"] for r in cfg["labels"].values()]
        if cfg.get("official_split_column"):
            needed.append(cfg["official_split_column"])
        missing_cols = [c for c in needed if c not in raw.columns]
        if missing_cols:
            raise ValueError(f"[{name}] {rel_path} 缺少列：{missing_cols}（现有列：{list(raw.columns)}）")
        out = pd.DataFrame({
            "text": raw[cfg["text_column"]].astype(object),
            "lang": cfg["lang"],
            "source": name,
        })
        # 关键步骤：未在映射表中声明的维度一律为 NaN
        mapped_mask = pd.Series(False, index=raw.index)
        for dim in LABEL_DIMENSIONS:
            if dim in cfg["labels"]:
                rule_cfg = cfg["labels"][dim]
                out[dim] = apply_label_rule(raw[rule_cfg["column"]], dim, rule_cfg, name)
                if rule_cfg["method"] == "mapped":
                    mapped_mask |= out[dim].notna()
            else:
                out[dim] = np.nan
        # 任一非空标签来自 mapped 规则 → 整行为 mapped
        out["label_method"] = np.where(mapped_mask, "mapped", "direct")
        if rel_path in split_by_file:
            out[OFFICIAL_SPLIT_COLUMN] = split_by_file[rel_path]
        elif cfg.get("official_split_column"):
            col = raw[cfg["official_split_column"]].astype(str).str.strip().str.lower()
            out[OFFICIAL_SPLIT_COLUMN] = col.map(OFFICIAL_SPLIT_ALIASES)
        else:
            out[OFFICIAL_SPLIT_COLUMN] = np.nan
        frames.append(out)
    if not frames:
        return None, inventory
    return pd.concat(frames, ignore_index=True), inventory


def build_unified(mapping):
    """映射所有数据源并合并为统一数据集；删除空文本行。返回 (unified, inventory DataFrame)。"""
    frames, inventory = [], []
    for name, cfg in iter_sources(mapping):
        df, inv = map_source(name, cfg)
        inventory.extend(inv)
        if df is not None:
            print(f"  [{name}] 映射 {len(df)} 行")
            frames.append(df)
    if not frames:
        raise RuntimeError("没有任何可用的原始数据文件，请检查 raw/ 与 label_mapping.json")
    unified = pd.concat(frames, ignore_index=True)
    text = unified["text"].astype(object)
    empty = text.isna() | (text.astype(str).str.strip() == "")
    if empty.any():
        print(f"  删除空文本行：{int(empty.sum())}")
    unified = unified.loc[~empty].reset_index(drop=True)
    unified["text"] = unified["text"].astype(str)
    return unified[UNIFIED_COLUMNS + [OFFICIAL_SPLIT_COLUMN]], pd.DataFrame(inventory)


def save_unified_csv(df, path):
    """保存统一数据；标签列转为可空整数，使缺失值写成空单元格而不是 0。"""
    out = df.copy()
    for dim in LABEL_DIMENSIONS:
        out[dim] = out[dim].astype("Int64")
    save_csv(out, path)


def read_unified_csv(path):
    """读取 unified / splits CSV，文本列按字符串读入，标签列为带 NaN 的浮点数。"""
    df = pd.read_csv(path, dtype={"text": str, "lang": str, "source": str, "label_method": str,
                                  OFFICIAL_SPLIT_COLUMN: str})
    df["text"] = df["text"].fillna("")
    for dim in LABEL_DIMENSIONS:
        df[dim] = pd.to_numeric(df[dim], errors="coerce").astype(float)
    return df


def mapping_distribution_report(df):
    """生成映射分布报告：每个 source 中各维度每个取值（含缺失）的数量与比例。"""
    rows = []
    for (lang, source), g in df.groupby(["lang", "source"], sort=True):
        n = len(g)
        for dim in LABEL_DIMENSIONS:
            values = SENTIMENT_LABELS if dim == "sentiment" else BINARY_LABEL_VALUES
            n_labeled = int(g[dim].notna().sum())
            for v in values:
                count = int((g[dim] == v).sum())
                rows.append({"lang": lang, "source": source, "dimension": dim, "value": str(v), "count": count,
                             "share_of_rows": count / n, "share_of_labeled": count / n_labeled if n_labeled else np.nan})
            missing = int(g[dim].isna().sum())
            rows.append({"lang": lang, "source": source, "dimension": dim, "value": "NA", "count": missing,
                         "share_of_rows": missing / n, "share_of_labeled": np.nan})
        for method in VALID_METHODS:
            count = int((g["label_method"] == method).sum())
            rows.append({"lang": lang, "source": source, "dimension": "label_method", "value": method,
                         "count": count, "share_of_rows": count / n, "share_of_labeled": np.nan})
    return pd.DataFrame(rows)


def write_mapping_docs(mapping, inventory, notes_path, inventory_path):
    """写出映射说明（docs/label_mapping_notes.md）与数据清单（docs/data_inventory.csv）。"""
    save_csv(inventory, inventory_path)
    lines = ["# 标签映射说明（自动生成）", "", f"生成时间：{utc_now_iso()}", "",
             "未在下表声明的维度保持为空值（NaN）。方法 direct = 人工标注（含对人工评分的确定性变换）；"
             "mapped = 由代理信号推得，不进入测试集。", ""]
    for name, cfg in iter_sources(mapping):
        lines += [f"## {name}（{cfg['lang']}）", "", f"- 文件：{', '.join(cfg['files'])}",
                  f"- 文本列：`{cfg['text_column']}`"]
        for dim, rule_cfg in cfg["labels"].items():
            extra = f"，value_map={rule_cfg['value_map']}" if rule_cfg["rule"] == "value_map" else ""
            lines.append(f"- {dim}：列 `{rule_cfg['column']}`，规则 `{rule_cfg['rule']}`{extra}，方法 `{rule_cfg['method']}`")
        if cfg.get("official_split_files"):
            lines.append(f"- 官方划分（按文件）：{cfg['official_split_files']}")
        if cfg.get("official_split_column"):
            lines.append(f"- 官方划分（按列）：`{cfg['official_split_column']}`")
        if cfg.get("notes"):
            lines.append(f"- 备注：{cfg['notes']}")
        lines.append("")
    lines += ["## EmoBank 取整", "", "sentiment = sign(x)·floor(|x|+0.5)，x = (valence−3)×2/2，结果裁剪到 [−2, 2]（远离零四舍五入）。", ""]
    assert_not_raw(notes_path)
    with open(notes_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

In [ ]:
# 阶段2：运行
label_mapping = load_label_mapping(FILES["label_mapping"])
unified, data_inventory = build_unified(label_mapping)
save_unified_csv(unified, FILES["unified"])
mapping_distribution = mapping_distribution_report(unified)
save_csv(mapping_distribution, FILES["mapping_distribution"])
write_mapping_docs(label_mapping, data_inventory, FILES["mapping_notes"], FILES["data_inventory"])

print(f"\n统一数据：{len(unified)} 行 → {FILES['unified']}")
print(unified.groupby(["lang", "source", "label_method"]).size().rename("rows").to_string())
print("\n情感倾向分布（含缺失）：")
print(mapping_distribution[mapping_distribution["dimension"] == "sentiment"]
      .pivot_table(index="source", columns="value", values="count", aggfunc="sum")
      .reindex(columns=[str(v) for v in SENTIMENT_LABELS] + ["NA"]).to_string())
print("\n压力表达倾向分布（含缺失）：")
print(mapping_distribution[mapping_distribution["dimension"] == "stress"]
      .pivot_table(index="source", columns="value", values="count", aggfunc="sum").to_string())
if not (unified["lang"] == LANG).any():
    print(f"\n⚠ 当前 LANG={LANG} 没有任何数据；请检查 label_mapping.json 中该语言的数据源。")

## 4. 阶段3：划分与泄露检查

- **输入**：`processed/unified.csv`
- **输出**：`splits/train.csv`、`splits/val.csv`、`splits/test.csv`、`reports/leakage_check.txt`
- 测试集只包含 `label_method == "direct"` 的样本（代码中有断言）。发现跨划分重复文本时报错并列出。

In [ ]:
# 阶段3：去重、分层与划分
def normalize_text(text):
    """文本规范化：去除首尾空格、合并内部空白、统一大小写（casefold）。"""
    return " ".join(str(text).split()).casefold()


def deduplicate(df):
    """按规范化文本去重（优先保留 direct 行，其余保持原顺序），返回 (去重后数据, 删除数量)。"""
    work = df.copy()
    work["_norm"] = work["text"].map(normalize_text)
    work["_priority"] = (work["label_method"] != "direct").astype(int)
    dup_rows = work[work.duplicated("_norm", keep=False)]
    if len(dup_rows):
        # 统计标签不一致的重复组（同一规范化文本、标签组合不同）
        n_groups = dup_rows["_norm"].nunique()
        n_conflict = int((label_signature(dup_rows).groupby(dup_rows["_norm"]).nunique() > 1).sum())
        print(f"  重复文本组：{n_groups}，其中标签不一致的组：{n_conflict}（保留优先级最高的第一条）")
    work = work.sort_values("_priority", kind="stable")
    kept = work.drop_duplicates("_norm", keep="first").sort_index()
    removed = len(work) - len(kept)
    return kept.drop(columns=["_norm", "_priority"]), removed


def label_signature(df):
    """生成标签组合签名，如 's=2|st=nan|ax=nan|dp=nan'，用于分层。"""
    parts = []
    for dim in LABEL_DIMENSIONS:
        col = df[dim].map(lambda v: "nan" if pd.isna(v) else str(int(v)))
        parts.append(SIGNATURE_ABBREVIATIONS[dim] + "=" + col)
    sig = parts[0]
    for p in parts[1:]:
        sig = sig + "|" + p
    return sig


def build_strata(df, min_size):
    """分层键 = lang|source|标签签名；小于 min_size 的层先并入 (lang,source) 的 rare 层，仍不足则并入全局 rare。"""
    keys = df["lang"] + "|" + df["source"] + "|" + label_signature(df)
    counts = keys.map(keys.value_counts())
    keys = keys.where(counts >= min_size, df["lang"] + "|" + df["source"] + "|rare")
    counts = keys.map(keys.value_counts())
    keys = keys.where(counts >= min_size, "rare")
    return keys


def stratified_assign(df, ratios, seed, min_size):
    """在每个层内按 ratios 随机分配划分名（确定性：固定种子、层按名称排序）；返回与 df 同索引的 Series。

    各层的目标数量采用“累积取整”：把小数部分结转到下一层，使总体比例贴近 ratios，
    避免大量小层逐层四舍五入后系统性地减少 val/test 样本。
    """
    result = pd.Series(index=df.index, dtype=object)
    if df.empty:
        return result
    strata = build_strata(df, min_size)
    rng = np.random.RandomState(seed)
    names = list(ratios)
    cum_target = {name: 0.0 for name in names[:-1]}
    cum_assigned = {name: 0 for name in names[:-1]}
    for key in sorted(strata.unique()):
        idx = strata.index[strata == key].to_numpy()
        idx = idx[rng.permutation(len(idx))]
        n = len(idx)
        counts = []
        for name in names[:-1]:
            cum_target[name] += n * ratios[name]
            c = max(0, int(round(cum_target[name])) - cum_assigned[name])
            c = min(c, n - sum(counts))  # 不超过层内剩余样本
            counts.append(c)
            cum_assigned[name] += c
        counts.append(n - sum(counts))
        start = 0
        for name, c in zip(names, counts):
            result.loc[idx[start:start + c]] = name
            start += c
    return result


def locked_languages():
    """返回已完成测试评估（存在测试标记文件）的语言列表。"""
    return [lang for lang in SUPPORTED_LANGS if os.path.exists(TEST_MARKER_TEMPLATE.format(lang=lang))]


def read_existing_split_assignment(lang):
    """读取 splits/ 中某语言现有的 {规范化文本: 划分名}；文件不存在时返回空字典。"""
    assignment = {}
    for name in ("train", "val", "test"):
        path = FILES[f"split_{name}"]
        if os.path.exists(path):
            df = read_unified_csv(path)
            for text in df.loc[df["lang"] == lang, "text"]:
                assignment[normalize_text(text)] = name
    return assignment


def assert_locked_splits_unchanged(split_df, langs):
    """对已完成测试评估的语言，确认新划分与 splits/ 中的现有划分完全一致，否则报错（防止已评估的测试集被改动）。"""
    for lang in langs:
        old = read_existing_split_assignment(lang)
        part = split_df[split_df["lang"] == lang]
        new = {normalize_text(t): s for t, s in zip(part["text"], part["split"])}
        if old != new:
            changed = len(set(old.items()) ^ set(new.items()))
            raise ValueError(f"语言 {lang} 已完成测试评估，但新的划分与现有 splits/ 不一致（{changed} 处差异）。"
                             f"请检查 raw/ 或映射表是否被改动；确需重新划分请设置 FORCE_RETEST = True。")
    return True


def split_dataset(unified, seed):
    """按规则划分：每种语言独立；官方划分优先；mapped 只进 train/val；返回带 split 列的数据。"""
    has_label = unified[LABEL_DIMENSIONS].notna().any(axis=1)
    if (~has_label).any():
        print(f"  删除无任何标签的行：{int((~has_label).sum())}")
    df, removed = deduplicate(unified.loc[has_label])
    print(f"  规范化文本去重删除：{removed} 行")
    df = df.copy()
    df["split"] = None
    official = df[OFFICIAL_SPLIT_COLUMN]
    for lang in sorted(df["lang"].unique()):
        in_lang = df["lang"] == lang
        direct = in_lang & (df["label_method"] == "direct")
        mapped = in_lang & (df["label_method"] == "mapped")
        # 1) 官方 test / val 直接沿用（仅 direct）
        df.loc[direct & (official == "test"), "split"] = "test"
        df.loc[direct & (official == "val"), "split"] = "val"
        # 2) 官方 train 再分为 train/val
        pool = direct & (official == "train")
        df.loc[pool, "split"] = stratified_assign(df.loc[pool], OFFICIAL_TRAIN_POOL_RATIOS, seed, MIN_STRATUM_SIZE)
        # 3) 无官方划分的 direct 行 70/15/15 分层
        free = direct & official.isna()
        df.loc[free, "split"] = stratified_assign(df.loc[free], SPLIT_RATIOS, seed, MIN_STRATUM_SIZE)
        # 4) mapped 行只进入 train/val
        df.loc[mapped, "split"] = stratified_assign(df.loc[mapped], MAPPED_SPLIT_RATIOS, seed, MIN_STRATUM_SIZE)
    assert df["split"].notna().all(), "存在未分配划分的样本"
    # 关键断言：测试集只能包含人工标注（direct）样本
    assert (df.loc[df["split"] == "test", "label_method"] == "direct").all(), "测试集中出现非 direct 样本！"
    return df

In [ ]:
# 阶段3：泄露检查
def leakage_check(splits, report_path):
    """检查规范化文本是否跨划分重复；写出报告，若有重复则抛出 ValueError 并列出。"""
    norm_sets = {name: set(df["text"].map(normalize_text)) for name, df in splits.items()}
    owners = {}
    for name, texts in norm_sets.items():
        for t in texts:
            owners.setdefault(t, []).append(name)
    duplicates = {t: names for t, names in owners.items() if len(names) > 1}
    lines = ["泄露检查报告", f"生成时间：{utc_now_iso()}",
             "规范化：去除首尾空格 + 合并内部空白 + casefold", ""]
    for name, df in splits.items():
        methods = df["label_method"].value_counts().to_dict()
        lines.append(f"{name}: {len(df)} 行；label_method = {methods}")
    test_df = splits.get("test")
    if test_df is not None:
        lines.append(f"测试集全部为 direct：{bool((test_df['label_method'] == 'direct').all())}")
    lines.append("")
    if duplicates:
        lines.append(f"结果：FAIL —— 发现 {len(duplicates)} 条跨划分重复文本：")
        for t, names in sorted(duplicates.items()):
            lines.append(f"  {names}: {t[:200]}")
    else:
        lines.append("结果：PASS —— 未发现跨划分重复文本。")
    assert_not_raw(report_path)
    with open(report_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines) + "\n")
    if duplicates:
        listing = "\n".join(f"  {names}: {t[:120]}" for t, names in list(duplicates.items())[:20])
        raise ValueError(f"发现 {len(duplicates)} 条跨划分重复文本（详见 {report_path}）：\n{listing}")
    return True

In [ ]:
# 阶段3：运行（该语言测试已评估过时拒绝重写 splits/）
SPLITS_WRITTEN = False
if os.path.exists(FILES["test_marker"]) and not FORCE_RETEST:
    print(f"⛔ 拒绝重写 splits/：{LANG} 的测试集已评估过（标记文件 {FILES['test_marker']}）。")
    print("   重写划分会使已报告的测试结果失效；确需重新划分请在配置区设置 FORCE_RETEST = True。")
else:
    unified = read_unified_csv(FILES["unified"])
    split_df = split_dataset(unified, SEED)
    if not FORCE_RETEST:
        # 其他已评估语言的划分必须保持不变
        assert_locked_splits_unchanged(split_df, [lang for lang in locked_languages() if lang != LANG])
    splits = {name: split_df[split_df["split"] == name].drop(columns=["split"]).reset_index(drop=True)
              for name in ("train", "val", "test")}
    leakage_check(splits, FILES["leakage_check"])
    assert (splits["test"]["label_method"] == "direct").all(), "测试集中出现非 direct 样本！"
    for name in ("train", "val", "test"):
        save_unified_csv(splits[name], FILES[f"split_{name}"])
    SPLITS_WRITTEN = True
    print("\n划分结果（行数）：")
    print(split_df.groupby(["lang", "source", "label_method", "split"]).size().unstack(fill_value=0).to_string())
    print(f"\n泄露检查：PASS → {FILES['leakage_check']}")

## 5. 阶段4：训练

- **输入**：`splits/train.csv`、`splits/val.csv`（只使用 `lang == LANG` 的行）
- **输出**：`models/{LANG}/model.pt`、`tokenizer/`、`encoder_config/`、`train_config.json`；追加 `reports/experiment_log.csv`
- 共享编码器 + 情感有序头（累积链接）+ 压力头（BCE），两个损失都带掩码；AdamW + 混合精度（bf16 优先，否则 fp16）；按验证集指标早停。

In [ ]:
# 阶段4：模型定义
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoConfig, AutoModel, AutoTokenizer


def inverse_softplus(x):
    """softplus 的反函数，用于把切点间隔初始值转换为无约束参数。"""
    x = np.asarray(x, dtype=np.float64)
    return np.log(np.expm1(x))


class OrdinalHead(nn.Module):
    """情感有序头（累积链接 / 比例优势模型）：输出一个标量分数 s，并维护 K−1 个递增切点。"""

    def __init__(self, hidden_size, init_cutpoints):
        """hidden_size：编码器隐藏维度；init_cutpoints：切点初始值（递增）。"""
        super().__init__()
        self.score = nn.Linear(hidden_size, 1)
        increments = np.diff(np.asarray(init_cutpoints, dtype=np.float64))
        # 切点参数化：第一个切点 + softplus(增量)，保证切点严格递增
        self.first_cut = nn.Parameter(torch.tensor(float(init_cutpoints[0])))
        self.raw_increments = nn.Parameter(torch.tensor(inverse_softplus(increments), dtype=torch.float32))

    def cutpoints(self):
        """返回递增的切点向量（长度 K−1）。"""
        steps = F.softplus(self.raw_increments)
        return torch.cat([self.first_cut.view(1), self.first_cut + torch.cumsum(steps, dim=0)])

    def forward(self, pooled):
        """输入池化向量，输出标量分数 s（形状 [B]）。"""
        return self.score(pooled).squeeze(-1)


def ordinal_probs(score, cutpoints, temperature=1.0):
    """由分数与切点计算 5 类概率：P(y ≤ k) = sigmoid((c_k − s)/T)，相邻差分得到各类概率。"""
    cdf = torch.sigmoid((cutpoints.unsqueeze(0) - score.unsqueeze(1)) / temperature)  # [B, K-1]
    ones = torch.ones_like(cdf[:, :1])
    zeros = torch.zeros_like(cdf[:, :1])
    probs = torch.cat([cdf, ones], dim=1) - torch.cat([zeros, cdf], dim=1)
    probs = probs.clamp_min(NUMERIC_EPS)
    return probs / probs.sum(dim=1, keepdim=True)


class TextExpressionModel(nn.Module):
    """共享编码器 + 情感有序头 + 压力二分类头（1 个 logit）。"""

    def __init__(self, encoder, init_cutpoints, dropout):
        """encoder：transformers 编码器；init_cutpoints：有序头切点初始值；dropout：头部 dropout。"""
        super().__init__()
        self.encoder = encoder
        hidden = encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.sentiment_head = OrdinalHead(hidden, init_cutpoints)
        self.stress_head = nn.Linear(hidden, 1)

    def forward(self, input_ids, attention_mask):
        """前向计算，返回情感分数、切点与压力 logit。"""
        hidden = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        pooled = self.dropout(hidden[:, 0].float())  # 取首个 token（<s>/[CLS]）作为句子表示
        # 两个输出头与切点计算在 fp32 下进行（关闭 autocast），保证有序概率的数值稳定
        with torch.autocast(device_type=pooled.device.type, enabled=False):
            return {
                "sent_score": self.sentiment_head(pooled),
                "cutpoints": self.sentiment_head.cutpoints(),
                "stress_logit": self.stress_head(pooled).squeeze(-1),
            }


def build_scratch_smoke_encoder(texts):
    """（仅 SMOKE_TEST）离线构建极小 RoBERTa 编码器与空白分词器，用于 HF Hub 不可达时的冒烟测试。"""
    from tokenizers import Tokenizer, models, normalizers, pre_tokenizers, processors
    from transformers import PreTrainedTokenizerFast, RobertaConfig, RobertaModel

    specials = ["<s>", "<pad>", "</s>", "<unk>", "<mask>"]
    normalizer = normalizers.BertNormalizer(clean_text=True, handle_chinese_chars=True, lowercase=True)
    vocab = {tok: i for i, tok in enumerate(specials)}
    for text in texts:
        for word in normalizer.normalize_str(str(text)).split():
            vocab.setdefault(word, len(vocab))
    tk = Tokenizer(models.WordLevel(vocab=vocab, unk_token="<unk>"))
    tk.normalizer = normalizer
    tk.pre_tokenizer = pre_tokenizers.WhitespaceSplit()
    tk.post_processor = processors.TemplateProcessing(
        single="<s> $A </s>", special_tokens=[("<s>", vocab["<s>"]), ("</s>", vocab["</s>"])])
    tokenizer = PreTrainedTokenizerFast(tokenizer_object=tk, bos_token="<s>", eos_token="</s>",
                                        pad_token="<pad>", unk_token="<unk>", mask_token="<mask>",
                                        cls_token="<s>", sep_token="</s>")
    config = RobertaConfig(vocab_size=len(vocab), max_position_embeddings=MAX_LENGTH + vocab["<pad>"] + 2,
                           pad_token_id=vocab["<pad>"], bos_token_id=vocab["<s>"], eos_token_id=vocab["</s>"],
                           **SMOKE_SCRATCH_ENCODER)
    return tokenizer, RobertaModel(config, add_pooling_layer=False), "scratch"


def load_encoder_and_tokenizer(name, fallback_texts=None):
    """加载预训练编码器与分词器；仅在 SMOKE_TEST 且下载失败时退回离线极小模型。返回 (tokenizer, encoder, 来源)。"""
    try:
        tokenizer = AutoTokenizer.from_pretrained(name)
        encoder = AutoModel.from_pretrained(name, add_pooling_layer=False)
        return tokenizer, encoder, "pretrained"
    except Exception as exc:  # noqa: BLE001
        if not SMOKE_TEST:
            raise
        print(f"⚠ 无法加载 {name}（{type(exc).__name__}），SMOKE_TEST 下改用离线极小模型。")
        return build_scratch_smoke_encoder(fallback_texts or [])

In [ ]:
# 阶段4：数据加载、掩码损失与评估指标
from sklearn.metrics import cohen_kappa_score, confusion_matrix, precision_recall_fscore_support
from torch.utils.data import DataLoader, Dataset


class TextLabelDataset(Dataset):
    """文本 + 情感/压力标签数据集；缺失标签保持为 NaN，由损失函数掩码处理。"""

    def __init__(self, df):
        """df：包含 text、sentiment、stress 列的 DataFrame。"""
        self.texts = df["text"].astype(str).tolist()
        self.sentiment = df["sentiment"].to_numpy(dtype=np.float32)
        self.stress = df["stress"].to_numpy(dtype=np.float32)

    def __len__(self):
        """返回样本数。"""
        return len(self.texts)

    def __getitem__(self, i):
        """返回 (文本, 情感标签, 压力标签)。"""
        return self.texts[i], self.sentiment[i], self.stress[i]


def make_collate_fn(tokenizer, max_length):
    """构造 collate 函数：批内动态填充、截断到 max_length，并把标签转为张量（NaN 保留）。"""
    def collate(batch):
        """把一批 (文本, 情感, 压力) 元组编码为模型输入张量与标签张量。"""
        texts, sent, stress = zip(*batch)
        enc = tokenizer(list(texts), padding=True, truncation=True, max_length=max_length,
                        return_tensors="pt", return_token_type_ids=False)
        enc["sentiment"] = torch.tensor(np.array(sent, dtype=np.float32))
        enc["stress"] = torch.tensor(np.array(stress, dtype=np.float32))
        return enc
    return collate


def make_loader(df, tokenizer, batch_size, shuffle, seed):
    """构造 DataLoader（shuffle 时使用固定种子的生成器，保证可复现）。"""
    generator = torch.Generator()
    generator.manual_seed(seed)
    return DataLoader(TextLabelDataset(df), batch_size=batch_size, shuffle=shuffle,
                      collate_fn=make_collate_fn(tokenizer, MAX_LENGTH), generator=generator)


def masked_ordinal_loss(score, cutpoints, labels, temperature=1.0):
    """情感有序损失：真实类别的负对数似然，只对有情感标签的样本计算（掩码）。"""
    mask = ~torch.isnan(labels)
    if mask.sum() == 0:
        return score.sum() * 0.0  # 保持计算图，贡献为 0
    probs = ordinal_probs(score[mask], cutpoints, temperature)
    target = (labels[mask] - min(SENTIMENT_LABELS)).long()
    return F.nll_loss(torch.log(probs), target)


def masked_bce_loss(logit, labels, temperature=1.0):
    """压力 BCE 损失，只对有压力标签的样本计算（掩码）。"""
    mask = ~torch.isnan(labels)
    if mask.sum() == 0:
        return logit.sum() * 0.0
    return F.binary_cross_entropy_with_logits(logit[mask] / temperature, labels[mask])


def get_amp_settings(device):
    """返回混合精度设置 (启用, dtype, 是否需要 GradScaler)：GPU 原生支持 bf16 时用 bf16，否则 fp16 + GradScaler；CPU 不启用。"""
    if device.type != "cuda":
        return False, torch.float32, False
    if bf16_native_supported():
        return True, torch.bfloat16, False
    return True, torch.float16, True


@torch.no_grad()
def predict_raw(model, loader, device):
    """在 loader 上做前向推理，返回情感分数、压力 logit（float32 numpy）与切点。"""
    model.eval()
    amp_enabled, amp_dtype, _ = get_amp_settings(device)
    scores, logits = [], []
    for batch in loader:
        with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=amp_enabled):
            out = model(batch["input_ids"].to(device), batch["attention_mask"].to(device))
        scores.append(out["sent_score"].float().cpu())
        logits.append(out["stress_logit"].float().cpu())
    cuts = model.sentiment_head.cutpoints().detach().float().cpu()
    return {"sent_score": torch.cat(scores).numpy(), "stress_logit": torch.cat(logits).numpy(),
            "cutpoints": cuts.numpy()}


def postprocess(raw, t_sent=1.0, t_stress=1.0):
    """把原始输出转换为情感概率、期望分数、情感标签与压力分数（应用温度）。"""
    probs = ordinal_probs(torch.tensor(raw["sent_score"]), torch.tensor(raw["cutpoints"]), t_sent).numpy()
    label_values = np.array(SENTIMENT_LABELS, dtype=np.float64)
    expected = np.clip(probs @ label_values, min(SENTIMENT_LABELS), max(SENTIMENT_LABELS))
    labels = label_values[np.argmax(probs, axis=1)].astype(int)
    stress_score = torch.sigmoid(torch.tensor(raw["stress_logit"]) / t_stress).numpy()
    return {"probs": probs, "expected": expected, "label": labels, "stress_score": stress_score}


def safe_qwk(y_true, y_pred):
    """计算情感 quadratic weighted kappa；无法定义（如只有单一类别）时返回 NaN。"""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        value = cohen_kappa_score(y_true, y_pred, labels=SENTIMENT_LABELS, weights="quadratic")
    return float(value) if np.isfinite(value) else float("nan")


def compute_sentiment_metrics(y_true, y_label, y_expected):
    """情感倾向指标：QWK、accuracy、MAE（基于 expected_score）、混淆矩阵；只用有标签样本。"""
    y_true = np.asarray(y_true, dtype=float)
    mask = ~np.isnan(y_true)
    if mask.sum() == 0:
        return {"n_labeled": 0, "qwk": None, "accuracy": None, "mae_expected": None, "confusion_matrix": None}
    yt = y_true[mask].astype(int)
    yl = np.asarray(y_label)[mask].astype(int)
    ye = np.asarray(y_expected, dtype=float)[mask]
    return {
        "n_labeled": int(mask.sum()),
        "qwk": safe_qwk(yt, yl),
        "accuracy": float((yt == yl).mean()),
        "mae_expected": float(np.abs(yt - ye).mean()),
        "confusion_matrix": {"labels": SENTIMENT_LABELS,
                             "matrix_rows_true_cols_pred": confusion_matrix(yt, yl, labels=SENTIMENT_LABELS).tolist()},
    }


def compute_stress_metrics(y_true, y_score, threshold):
    """压力表达倾向指标：precision、recall、F1、正例数量；只用有标签样本。"""
    y_true = np.asarray(y_true, dtype=float)
    mask = ~np.isnan(y_true)
    if mask.sum() == 0:
        return {"n_labeled": 0, "n_positive": 0, "precision": None, "recall": None, "f1": None,
                "threshold": float(threshold)}
    yt = y_true[mask].astype(int)
    yp = (np.asarray(y_score, dtype=float)[mask] >= threshold).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(yt, yp, average="binary", pos_label=1, zero_division=0)
    return {"n_labeled": int(mask.sum()), "n_positive": int(yt.sum()), "n_predicted_positive": int(yp.sum()),
            "precision": float(p), "recall": float(r), "f1": float(f1), "threshold": float(threshold)}

In [ ]:
# 阶段4：训练循环、早停、模型保存与实验日志
import copy

from transformers import get_linear_schedule_with_warmup

EXPERIMENT_LOG_COLUMNS = ["timestamp_utc", "lang", "encoder", "encoder_source", "smoke_test", "n_train", "n_val",
                          "learning_rate", "head_learning_rate", "epochs_run", "best_epoch", "best_val_metric",
                          "val_sentiment_qwk", "val_stress_f1", "val_stress_recall", "train_seconds", "gpu",
                          "precision", "model_sha256", "config_json"]


def early_stop_metric(sent_metrics, stress_metrics):
    """早停指标 = w1 × 情感 QWK + w2 × 压力 F1；某维度在验证集无标签时该项记 0 并警告。"""
    total = 0.0
    qwk = sent_metrics.get("qwk")
    f1 = stress_metrics.get("f1")
    if sent_metrics["n_labeled"] == 0:
        print("  ⚠ 验证集没有情感标签，早停指标中情感项记为 0")
    else:
        total += EARLY_STOP_WEIGHTS["sentiment_qwk"] * (0.0 if qwk is None or np.isnan(qwk) else qwk)
    if stress_metrics["n_labeled"] == 0:
        print("  ⚠ 验证集没有压力标签，早停指标中压力项记为 0")
    else:
        total += EARLY_STOP_WEIGHTS["stress_f1"] * (f1 or 0.0)
    return total


def evaluate_on_loader(model, loader, df, device):
    """在验证集上评估（温度 = 1，压力阈值 = EARLY_STOP_STRESS_THRESHOLD），返回 (情感指标, 压力指标, 早停指标)。"""
    post = postprocess(predict_raw(model, loader, device))
    sent = compute_sentiment_metrics(df["sentiment"].to_numpy(), post["label"], post["expected"])
    stress = compute_stress_metrics(df["stress"].to_numpy(), post["stress_score"], EARLY_STOP_STRESS_THRESHOLD)
    return sent, stress, early_stop_metric(sent, stress)


def hyperparameter_snapshot():
    """收集本次训练使用的全部配置，写入 train_config.json 与实验日志。"""
    return {"lang": LANG, "encoder": ENCODER_NAME, "encoder_variant": ENCODER_VARIANT, "learning_rate": LEARNING_RATE,
            "head_learning_rate": HEAD_LEARNING_RATE,
            "batch_size": BATCH_SIZE, "grad_accum_steps": GRAD_ACCUM_STEPS,
            "effective_batch_size": BATCH_SIZE * GRAD_ACCUM_STEPS, "max_epochs": MAX_EPOCHS,
            "warmup_ratio": WARMUP_RATIO, "weight_decay": WEIGHT_DECAY, "max_length": MAX_LENGTH,
            "max_grad_norm": MAX_GRAD_NORM, "dropout": DROPOUT, "w_sent": W_SENT, "w_stress": W_STRESS,
            "seed": SEED, "early_stop_patience": EARLY_STOP_PATIENCE, "early_stop_weights": EARLY_STOP_WEIGHTS,
            "early_stop_stress_threshold": EARLY_STOP_STRESS_THRESHOLD, "sentiment_labels": SENTIMENT_LABELS,
            "ordinal_init_cutpoints": ORDINAL_INIT_CUTPOINTS, "smoke_test": SMOKE_TEST}


def save_model_artifacts(model, tokenizer, train_config):
    """把模型权重、分词器、编码器结构配置与训练配置保存到 models/{LANG}/（不写出该目录之外）。"""
    for key in ("model_weights", "train_config", "tokenizer_dir", "encoder_config_dir"):
        assert os.path.realpath(FILES[key]).startswith(os.path.realpath(MODEL_DIR)), "模型产物必须位于 models/{LANG}/"
    os.makedirs(MODEL_DIR, exist_ok=True)
    torch.save(model.state_dict(), FILES["model_weights"])
    tokenizer.save_pretrained(FILES["tokenizer_dir"])
    model.encoder.config.save_pretrained(FILES["encoder_config_dir"])
    save_json(train_config, FILES["train_config"])


def append_experiment_log(record, path):
    """把一次训练的配置、指标、时长、GPU 型号追加写入 reports/experiment_log.csv。"""
    row = pd.DataFrame([{col: record.get(col) for col in EXPERIMENT_LOG_COLUMNS}])
    assert_not_raw(path)
    if os.path.exists(path):
        old = pd.read_csv(path)
        if list(old.columns) != EXPERIMENT_LOG_COLUMNS:
            # 旧日志的列与当前不同（例如新增了列）：合并后整体重写，保证列对齐
            cols = EXPERIMENT_LOG_COLUMNS + [c for c in old.columns if c not in EXPERIMENT_LOG_COLUMNS]
            save_csv(pd.concat([old, row], ignore_index=True).reindex(columns=cols), path)
            return
    row.to_csv(path, mode="a", header=not os.path.exists(path), index=False, encoding="utf-8")


def split_rows_sha256(path, lang):
    """计算某划分文件中该语言行的 SHA-256（规范化 CSV 序列化），用于判断训练所用划分是否变化。

    只对该语言的行求哈希，因此加入另一种语言的数据不会让已训练的模型被判定为“划分已变化”。
    """
    df = read_unified_csv(path)
    df = df[df["lang"] == lang][UNIFIED_COLUMNS + [OFFICIAL_SPLIT_COLUMN]].reset_index(drop=True)
    return hashlib.sha256(df.to_csv(index=False).encode("utf-8")).hexdigest()


def existing_model_status():
    """判断 models/{LANG}/ 中的模型能否直接复用；返回 (可复用, 原因)。原因为空字符串表示可复用。"""
    if not os.path.exists(FILES["model_weights"]):
        return False, "没有已训练的模型"
    cfg = load_json(FILES["train_config"])
    if cfg is None:
        return False, "缺少配置（train_config.json 不存在）"
    if not cfg.get("training_complete", False):
        return False, "未完成训练（上次训练可能中途断开，model.pt 只是中间检查点）"
    if (cfg.get("train_csv_sha256") != split_rows_sha256(FILES["split_train"], LANG) or
            cfg.get("val_csv_sha256") != split_rows_sha256(FILES["split_val"], LANG)):
        return False, "划分已变化（splits/ 中该语言的 train / val 与训练时不同）"
    return True, ""


def load_trained_model(device):
    """从 models/{LANG}/ 重建编码器结构并加载权重与分词器，返回 (model, tokenizer, train_config)。"""
    if not os.path.exists(FILES["model_weights"]):
        raise FileNotFoundError(f"找不到 {FILES['model_weights']}，请先运行阶段4（训练）。")
    train_config = load_json(FILES["train_config"])
    encoder = AutoModel.from_config(AutoConfig.from_pretrained(FILES["encoder_config_dir"]), add_pooling_layer=False)
    model = TextExpressionModel(encoder, train_config["ordinal_init_cutpoints"], train_config["dropout"])
    state = torch.load(FILES["model_weights"], map_location=device, weights_only=True)
    model.load_state_dict(state)
    model.to(device).eval()
    tokenizer = AutoTokenizer.from_pretrained(FILES["tokenizer_dir"])
    return model, tokenizer, train_config


def train_model(train_df, val_df):
    """训练共享编码器 + 双头模型：AdamW、混合精度、梯度累积、线性 warmup、按验证集指标早停；返回训练摘要。"""
    set_seed(SEED)  # 关键步骤：训练前设置所有随机种子
    # 记录本次训练所用划分（该语言行）的哈希，训练完成后写入 train_config.json
    split_hashes = {"train_csv_sha256": split_rows_sha256(FILES["split_train"], LANG),
                    "val_csv_sha256": split_rows_sha256(FILES["split_val"], LANG)}
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type == "cuda" and ENCODER_VARIANT == "large" and LANG == "en":
        mem_gb = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3
        if mem_gb < LARGE_MODEL_MIN_GPU_MEM_GB:
            print(f"⚠ 显存 {mem_gb:.1f} GB 低于 {LARGE_MODEL_MIN_GPU_MEM_GB} GB，roberta-large 可能显存不足。")
    tokenizer, encoder, encoder_source = load_encoder_and_tokenizer(ENCODER_NAME, train_df["text"].tolist())
    model = TextExpressionModel(encoder, ORDINAL_INIT_CUTPOINTS, DROPOUT).to(device)

    # 只保留至少有情感或压力标签之一的训练样本（其余样本两个损失都被掩码）
    train_df = train_df[train_df[["sentiment", "stress"]].notna().any(axis=1)].reset_index(drop=True)
    train_loader = make_loader(train_df, tokenizer, BATCH_SIZE, shuffle=True, seed=SEED)
    val_loader = make_loader(val_df, tokenizer, EVAL_BATCH_SIZE, shuffle=False, seed=SEED)

    # AdamW 参数组：编码器用 LEARNING_RATE，两个输出头（含切点）用 HEAD_LEARNING_RATE；
    # 偏置、LayerNorm 与有序切点不做权重衰减
    head_param_ids = {id(p) for m in (model.sentiment_head, model.stress_head) for p in m.parameters()}
    no_decay = ("bias", "LayerNorm.weight", "layer_norm.weight", "first_cut", "raw_increments")
    groups = {("enc", True): [], ("enc", False): [], ("head", True): [], ("head", False): []}
    for n, p in model.named_parameters():
        part = "head" if id(p) in head_param_ids else "enc"
        groups[(part, not any(nd in n for nd in no_decay))].append(p)
    lr_of = {"enc": LEARNING_RATE, "head": HEAD_LEARNING_RATE}
    params = [{"params": ps, "lr": lr_of[part], "weight_decay": WEIGHT_DECAY if decay else 0.0}
              for (part, decay), ps in groups.items() if ps]
    optimizer = torch.optim.AdamW(params, lr=LEARNING_RATE)
    steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
    total_steps = steps_per_epoch * MAX_EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP_RATIO * total_steps), total_steps)
    amp_enabled, amp_dtype, use_scaler = get_amp_settings(device)
    scaler = torch.amp.GradScaler("cuda", enabled=use_scaler)  # 仅 fp16 时启用
    precision_name = str(amp_dtype).replace("torch.", "") if amp_enabled else "fp32"
    print(f"设备：{device} | 精度：{precision_name} | 训练样本：{len(train_df)} | 验证样本：{len(val_df)} | "
          f"学习率：编码器 {LEARNING_RATE}，输出头 {HEAD_LEARNING_RATE}")

    best = {"metric": -float("inf"), "epoch": None, "sent": None, "stress": None}
    patience_left = EARLY_STOP_PATIENCE
    epochs_run = 0
    start_time = time.time()
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running = []
        for step, batch in enumerate(train_loader):
            with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=amp_enabled):
                out = model(batch["input_ids"].to(device), batch["attention_mask"].to(device))
            # 关键步骤：损失在 float32 下计算，缺失标签被掩码
            loss_sent = masked_ordinal_loss(out["sent_score"].float(), out["cutpoints"].float(),
                                            batch["sentiment"].to(device))
            loss_stress = masked_bce_loss(out["stress_logit"].float(), batch["stress"].to(device))
            loss = W_SENT * loss_sent + W_STRESS * loss_stress
            scaler.scale(loss / GRAD_ACCUM_STEPS).backward()
            running.append(loss.item())
            if (step + 1) % GRAD_ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
        epochs_run = epoch
        sent, stress, metric = evaluate_on_loader(model, val_loader, val_df, device)
        qwk_str = "n/a" if sent["qwk"] is None else f"{sent['qwk']:.4f}"
        f1_str = "n/a" if stress["f1"] is None else f"{stress['f1']:.4f}"
        rec_str = "n/a" if stress["recall"] is None else f"{stress['recall']:.4f}"
        print(f"Epoch {epoch}/{MAX_EPOCHS} | train loss {np.mean(running):.4f} | 情感倾向 QWK {qwk_str} | "
              f"压力表达倾向 F1 {f1_str} | recall {rec_str} | 早停指标 {metric:.4f}")
        if metric > best["metric"]:
            best = {"metric": metric, "epoch": epoch, "sent": sent, "stress": stress}
            patience_left = EARLY_STOP_PATIENCE
            train_config = hyperparameter_snapshot()
            train_config.update({"training_complete": False,  # 训练结束后才改为 true
                                 "encoder_source": encoder_source, "best_epoch": epoch, "best_val_metric": metric,
                                 "val_sentiment": sent, "val_stress": stress, "saved_at": utc_now_iso(),
                                 "note": "Outputs are textual tendency estimates only."})
            save_model_artifacts(model, tokenizer, train_config)  # 保存当前最佳模型
            print(f"  ✓ 新的最佳模型已保存到 {MODEL_DIR}")
        else:
            patience_left -= 1
            if patience_left <= 0:
                print(f"  早停：连续 {EARLY_STOP_PATIENCE} 个 epoch 验证集指标未提升")
                break
    train_seconds = time.time() - start_time
    # 关键步骤：训练正常结束（早停或最后一个 epoch）后才标记为完整训练，并记录所用划分的哈希
    final_config = load_json(FILES["train_config"])
    final_config.update({"training_complete": True, **split_hashes, "epochs_run": epochs_run,
                         "completed_at": utc_now_iso()})
    save_json(final_config, FILES["train_config"])
    summary = {
        "timestamp_utc": utc_now_iso(), "lang": LANG, "encoder": ENCODER_NAME, "encoder_source": encoder_source,
        "smoke_test": SMOKE_TEST, "n_train": len(train_df), "n_val": len(val_df),
        "learning_rate": LEARNING_RATE, "head_learning_rate": HEAD_LEARNING_RATE, "epochs_run": epochs_run,
        "best_epoch": best["epoch"], "best_val_metric": best["metric"],
        "val_sentiment_qwk": best["sent"]["qwk"], "val_stress_f1": best["stress"]["f1"],
        "val_stress_recall": best["stress"]["recall"], "train_seconds": round(train_seconds, 1),
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
        "precision": precision_name, "model_sha256": model_fingerprint(),
        "config_json": json.dumps(to_jsonable(hyperparameter_snapshot()), ensure_ascii=False),
    }
    append_experiment_log(summary, FILES["experiment_log"])
    return summary

In [ ]:
# 阶段4：运行
train_split = read_unified_csv(FILES["split_train"])
val_split = read_unified_csv(FILES["split_val"])
train_split = train_split[train_split["lang"] == LANG].reset_index(drop=True)
val_split = val_split[val_split["lang"] == LANG].reset_index(drop=True)
assert len(train_split) > 0, f"LANG={LANG} 没有训练数据"
assert len(val_split) > 0, f"LANG={LANG} 没有验证数据"
MODEL_REUSABLE, MODEL_STATUS_REASON = existing_model_status()
if SKIP_TRAINING_IF_MODEL_EXISTS and MODEL_REUSABLE:
    # 已有模型：加载而不重新训练，避免“全部运行”时重训导致校准 / 测试结果与模型不一致
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model, tokenizer, train_config = load_trained_model(device)
    TRAIN_SUMMARY = {"skipped_training": True, "lang": LANG, "encoder": train_config["encoder"],
                     "best_epoch": train_config.get("best_epoch"), "saved_at": train_config.get("saved_at"),
                     "model_sha256": model_fingerprint()}
    print(f"⏭ 已存在完整训练的 {FILES['model_weights']}（划分未变化），跳过训练并加载已有模型。")
    print("   如需重新训练，请设置 SKIP_TRAINING_IF_MODEL_EXISTS = False（重新训练后必须重新运行阶段5）。")
    if train_config["encoder"] != ENCODER_NAME:
        print(f"⚠ 已有模型的编码器为 {train_config['encoder']}，与当前配置 {ENCODER_NAME} 不同。")
else:
    if SKIP_TRAINING_IF_MODEL_EXISTS and os.path.exists(FILES["model_weights"]):
        print(f"⚠ 不复用已有模型：{MODEL_STATUS_REASON}。将重新训练（之后需重新运行阶段5）。")
    TRAIN_SUMMARY = train_model(train_split, val_split)
    TRAIN_SUMMARY["skipped_training"] = False
print(json.dumps(to_jsonable({k: v for k, v in TRAIN_SUMMARY.items() if k != "config_json"}),
                 ensure_ascii=False, indent=2))

## 6. 阶段5：校准与阈值

- **输入**：`splits/val.csv`（只用验证集）、`models/{LANG}/`
- **输出**：`configs/temperature.json`（`T_sent`、`T_stress`）、`configs/thresholds.json`（压力阈值）
- 温度缩放：在冻结的验证集输出上用 LBFGS 最小化 NLL。压力阈值在 0.05–0.95（步长 0.01）中搜索使 F1 最大的值。情感输出不使用阈值。

In [ ]:
# 阶段5：校准结果读取与模型指纹一致性检查（load_trained_model 定义在阶段4 的单元中）
def load_calibration(lang, model_sha256):
    """读取该语言的温度与压力阈值，并核对它们是用当前 model.pt（SHA-256 指纹）拟合的；不一致时提示重新运行阶段5。"""
    temps = (load_json(FILES["temperature"], {}) or {}).get(lang)
    thresholds = (load_json(FILES["thresholds"], {}) or {}).get(lang)
    if temps is None or thresholds is None:
        raise FileNotFoundError(f"configs 中缺少 {lang} 的温度或阈值，请先运行阶段5。")
    for name, entry in (("temperature.json", temps), ("thresholds.json", thresholds)):
        recorded = entry.get("model_sha256")
        if recorded != model_sha256:
            raise RuntimeError(
                f"configs/{name} 中 {lang} 的校准结果与当前模型不一致"
                f"（当前模型指纹 {str(model_sha256)[:12]}…，校准记录 {str(recorded)[:12]}…）。"
                f"模型可能已重新训练，请重新运行阶段5（校准与阈值）。")
    return float(temps["T_sent"]), float(temps["T_stress"]), float(thresholds["stress"])

In [ ]:
# 阶段5：温度缩放与阈值搜索
def _fit_temperature(nll_fn):
    """通用温度拟合：优化 log T（保证 T > 0），LBFGS 最小化 nll_fn(T)，结果裁剪到 TEMPERATURE_BOUNDS。"""
    log_t = torch.zeros(1, requires_grad=True)
    optimizer = torch.optim.LBFGS([log_t], lr=TEMPERATURE_LBFGS_LR, max_iter=TEMPERATURE_LBFGS_MAX_ITER,
                                  line_search_fn="strong_wolfe")

    def closure():
        """LBFGS 闭包：计算当前温度下的 NLL 并反向传播。"""
        optimizer.zero_grad()
        loss = nll_fn(log_t.exp())
        loss.backward()
        return loss

    optimizer.step(closure)
    t = float(np.clip(log_t.exp().item(), *TEMPERATURE_BOUNDS))
    return t


def fit_sentiment_temperature(raw, labels):
    """在验证集上拟合情感温度 T_sent（作用于 (c_k − s)/T）；无标签时返回 1.0。"""
    mask = ~np.isnan(labels)
    if mask.sum() == 0:
        print("⚠ 验证集没有情感标签，T_sent 使用 1.0")
        return 1.0, None, None
    score = torch.tensor(raw["sent_score"][mask])
    cuts = torch.tensor(raw["cutpoints"])
    y = torch.tensor(labels[mask])
    nll = lambda t: masked_ordinal_loss(score, cuts, y, t)
    before = float(nll(torch.tensor(1.0)))
    t = _fit_temperature(nll)
    return t, before, float(nll(torch.tensor(t)))


def fit_stress_temperature(raw, labels):
    """在验证集上拟合压力温度 T_stress（logit / T）；无标签时返回 1.0。"""
    mask = ~np.isnan(labels)
    if mask.sum() == 0:
        print("⚠ 验证集没有压力标签，T_stress 使用 1.0")
        return 1.0, None, None
    logit = torch.tensor(raw["stress_logit"][mask])
    y = torch.tensor(labels[mask].astype(np.float32))
    nll = lambda t: masked_bce_loss(logit, y, t)
    before = float(nll(torch.tensor(1.0)))
    t = _fit_temperature(nll)
    return t, before, float(nll(torch.tensor(t)))


def threshold_grid():
    """返回压力阈值搜索网格（来自 THRESHOLD_SEARCH，含端点）。"""
    s = THRESHOLD_SEARCH
    n = int(round((s["stop"] - s["start"]) / s["step"])) + 1
    return np.round(s["start"] + s["step"] * np.arange(n), 6)


def search_stress_threshold(labels, scores):
    """在验证集上搜索使压力 F1 最大的阈值（并列时取较小阈值）；返回 (阈值, F1)。"""
    mask = ~np.isnan(labels)
    if mask.sum() == 0:
        print(f"⚠ 验证集没有压力标签，阈值使用 {EARLY_STOP_STRESS_THRESHOLD}")
        return float(EARLY_STOP_STRESS_THRESHOLD), None
    best_t, best_f1 = None, -1.0
    for t in threshold_grid():
        f1 = compute_stress_metrics(labels, scores, t)["f1"]
        if f1 > best_f1 + 1e-12:
            best_t, best_f1 = float(t), f1
    return best_t, best_f1

In [ ]:
# 阶段5：运行（只使用验证集；该语言测试已评估过时拒绝重新校准）
CALIBRATION_WRITTEN = False
if os.path.exists(FILES["test_marker"]) and not FORCE_RETEST:
    print(f"⛔ 拒绝重新校准：{LANG} 的测试集已评估过（标记文件 {FILES['test_marker']}）。")
    print("   已报告的测试结果是用现有温度与阈值得到的，重新拟合会使测试结果与 configs/ 中的校准参数不一致。")
    print("   确需重新校准请设置 FORCE_RETEST = True，并在之后重新运行阶段6。")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model, tokenizer, train_config = load_trained_model(device)
    MODEL_SHA256 = model_fingerprint()  # 记录校准所用模型的指纹，阶段6 / 7 据此核对
    val_split = read_unified_csv(FILES["split_val"])
    val_split = val_split[val_split["lang"] == LANG].reset_index(drop=True)
    val_raw = predict_raw(model, make_loader(val_split, tokenizer, EVAL_BATCH_SIZE, False, SEED), device)

    t_sent, nll_sent_before, nll_sent_after = fit_sentiment_temperature(val_raw, val_split["sentiment"].to_numpy())
    t_stress, nll_stress_before, nll_stress_after = fit_stress_temperature(val_raw, val_split["stress"].to_numpy())
    update_lang_entry(FILES["temperature"], LANG, {
        "T_sent": t_sent, "T_stress": t_stress, "fitted_on": "val", "n_val": len(val_split),
        "nll_sent_before": nll_sent_before, "nll_sent_after": nll_sent_after,
        "nll_stress_before": nll_stress_before, "nll_stress_after": nll_stress_after,
        "model": train_config["encoder"], "model_sha256": MODEL_SHA256, "fitted_at": utc_now_iso()})

    val_post = postprocess(val_raw, t_sent, t_stress)
    stress_threshold, val_best_f1 = search_stress_threshold(val_split["stress"].to_numpy(), val_post["stress_score"])
    update_lang_entry(FILES["thresholds"], LANG, {
        "stress": stress_threshold, "val_f1": val_best_f1, "search": THRESHOLD_SEARCH, "objective": "max F1",
        "fitted_on": "val", "applies_to": "calibrated stress score (sigmoid(logit / T_stress))",
        "sentiment": None, "model_sha256": MODEL_SHA256, "fitted_at": utc_now_iso()})
    print(f"T_sent = {t_sent:.4f}（NLL {nll_sent_before} → {nll_sent_after}）")
    print(f"T_stress = {t_stress:.4f}（NLL {nll_stress_before} → {nll_stress_after}）")
    print(f"压力表达倾向阈值 = {stress_threshold}（验证集 F1 = {val_best_f1}）；情感倾向不使用阈值")
    CALIBRATION_WRITTEN = True

## 7. 阶段6：测试评估与误差分析（只运行一次）

- **输入**：`splits/test.csv`、`models/{LANG}/`、`configs/temperature.json`、`configs/thresholds.json`
- **输出**：`reports/test_results.json`、`reports/error_cases.csv`（含空白 `error_type` 列供人工填写）、`reports/baseline_results.json`
- 成功后写入标记文件 `reports/.test_evaluated_{LANG}`；再次运行会被拒绝，除非 `FORCE_RETEST = True`。

In [ ]:
# 阶段6：评估与误差分析函数
def predict_dataframe(model, tokenizer, df, device, t_sent, t_stress, threshold):
    """对 DataFrame 打分，返回附带预测列（情感标签、期望分数、各类概率、压力分数与标签）的副本。"""
    raw = predict_raw(model, make_loader(df, tokenizer, EVAL_BATCH_SIZE, False, SEED), device)
    post = postprocess(raw, t_sent, t_stress)
    out = df.copy()
    out["pred_sentiment_label"] = post["label"]
    out["pred_expected_score"] = post["expected"]
    for j, v in enumerate(SENTIMENT_LABELS):
        out[f"prob_{v}"] = post["probs"][:, j]
    out["pred_stress_score"] = post["stress_score"]
    out["pred_stress_label"] = (post["stress_score"] >= threshold).astype(int)
    return out


def metrics_for(pred_df, threshold, include_stress=True):
    """计算一个子集的情感与压力指标。"""
    result = {"n": len(pred_df),
              "sentiment": compute_sentiment_metrics(pred_df["sentiment"].to_numpy(),
                                                     pred_df["pred_sentiment_label"].to_numpy(),
                                                     pred_df["pred_expected_score"].to_numpy())}
    if include_stress:
        result["stress"] = compute_stress_metrics(pred_df["stress"].to_numpy(),
                                                  pred_df["pred_stress_score"].to_numpy(), threshold)
    else:
        result["stress"] = None
        result["stress_note"] = BASELINE_STRESS_NOTE
    return result


def evaluate_predictions(pred_df, threshold, include_stress=True):
    """总体指标，以及按 lang、按 source 分别报告的指标。"""
    return {
        "overall": metrics_for(pred_df, threshold, include_stress),
        "by_lang": {k: metrics_for(g, threshold, include_stress) for k, g in pred_df.groupby("lang")},
        "by_source": {k: metrics_for(g, threshold, include_stress) for k, g in pred_df.groupby("source")},
    }


def select_error_cases(pred_df, n):
    """选出预测错误最严重的 n 条：严重度 = |情感真值 − 期望分数|/4 + |压力真值 − 压力分数|（只计有标签维度）。"""
    span = max(SENTIMENT_LABELS) - min(SENTIMENT_LABELS)
    sent_err = (pred_df["sentiment"] - pred_df["pred_expected_score"]).abs() / span
    stress_err = (pred_df["stress"] - pred_df["pred_stress_score"]).abs()
    severity = sent_err.fillna(0.0) + stress_err.fillna(0.0)
    wrong = ((pred_df["sentiment"].notna() & (pred_df["sentiment"] != pred_df["pred_sentiment_label"])) |
             (pred_df["stress"].notna() & (pred_df["stress"] != pred_df["pred_stress_label"])))
    cases = pred_df.assign(severity=severity).loc[wrong].sort_values("severity", ascending=False).head(n)
    cases = cases.rename(columns={"sentiment": "true_sentiment", "stress": "true_stress"})
    cases["error_type"] = ""  # 空白列，供人工填写误差类型
    cols = ["text", "lang", "source", "true_sentiment", "pred_sentiment_label", "pred_expected_score",
            "true_stress", "pred_stress_score", "pred_stress_label", "severity", "error_type"]
    return cases[cols]


def save_error_cases(cases, path, lang):
    """保存误差案例：替换该语言旧的案例，保留其他语言的案例。"""
    if os.path.exists(path):
        old = pd.read_csv(path, dtype={"text": str, "lang": str})
        cases = pd.concat([old[old["lang"] != lang], cases], ignore_index=True)
    for col in ("true_sentiment", "pred_sentiment_label", "true_stress", "pred_stress_label"):
        cases[col] = pd.to_numeric(cases[col], errors="coerce").astype("Int64")
    save_csv(cases, path)

In [ ]:
# 阶段6：运行测试评估（只运行一次；受标记文件保护，标记在下一单元的零样本基线完成后写入）
TEST_EVALUATED_NOW = False
RUN_TEST_EVALUATION = False
if os.path.exists(FILES["test_marker"]) and not FORCE_RETEST:
    print(f"⛔ 拒绝运行：{LANG} 的测试集已评估过（标记文件 {FILES['test_marker']}）。")
    print("   测试集只应评估一次；如确需重新评估，请在配置区设置 FORCE_RETEST = True。")
else:
    test_split = read_unified_csv(FILES["split_test"])
    test_split = test_split[test_split["lang"] == LANG].reset_index(drop=True)
    assert len(test_split) > 0, f"LANG={LANG} 没有测试数据"
    # 关键断言：测试集只能包含人工标注（direct）样本
    assert (test_split["label_method"] == "direct").all(), "测试集中出现非 direct 样本！"
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model, tokenizer, train_config = load_trained_model(device)
    TEST_MODEL_SHA256 = model_fingerprint()
    t_sent, t_stress, stress_threshold = load_calibration(LANG, TEST_MODEL_SHA256)  # 核对校准与模型指纹
    test_pred = predict_dataframe(model, tokenizer, test_split, device, t_sent, t_stress, stress_threshold)
    test_results = evaluate_predictions(test_pred, stress_threshold)
    test_results["meta"] = {"lang": LANG, "model": train_config["encoder"], "n_test": len(test_pred),
                            "T_sent": t_sent, "T_stress": t_stress, "stress_threshold": stress_threshold,
                            "evaluated_at": utc_now_iso(), "smoke_test": SMOKE_TEST,
                            "model_sha256": TEST_MODEL_SHA256,
                            "note": "情感倾向 / 压力表达倾向的文本估计指标"}
    update_lang_entry(FILES["test_results"], LANG, test_results)
    error_cases = select_error_cases(test_pred, N_ERROR_CASES)
    save_error_cases(error_cases, FILES["error_cases"], LANG)
    RUN_TEST_EVALUATION = True  # 下一单元运行零样本基线并写入测试标记
    overall = test_results["overall"]
    print(f"测试集 {len(test_pred)} 行")
    print("情感倾向：", json.dumps(to_jsonable({k: v for k, v in overall["sentiment"].items()
                                              if k != "confusion_matrix"}), ensure_ascii=False))
    print("压力表达倾向：", json.dumps(to_jsonable(overall["stress"]), ensure_ascii=False))
    print(f"误差案例 {len(error_cases)} 条 → {FILES['error_cases']}")

In [ ]:
# 阶段6：零样本基线（预训练情感模型；加载失败只记录并跳过，不中断流程）
from transformers import AutoModelForSequenceClassification


def run_zero_shot_baseline(test_df, baseline_cfg):
    """用预训练情感模型对测试集打分，输出与主模型相同的情感指标；压力指标为 null。失败时返回 skipped 记录。"""
    try:
        tok = AutoTokenizer.from_pretrained(baseline_cfg["name"])
        clf = AutoModelForSequenceClassification.from_pretrained(baseline_cfg["name"])
        values = np.asarray(baseline_cfg["label_values"], dtype=np.float64)
        if clf.config.num_labels != len(values):
            raise ValueError(f"模型输出 {clf.config.num_labels} 类，但 label_values 有 {len(values)} 个")
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        clf.to(device).eval()
        probs = []
        texts = test_df["text"].astype(str).tolist()
        with torch.no_grad():
            for i in range(0, len(texts), EVAL_BATCH_SIZE):
                enc = tok(texts[i:i + EVAL_BATCH_SIZE], padding=True, truncation=True, max_length=MAX_LENGTH,
                          return_tensors="pt").to(device)
                probs.append(torch.softmax(clf(**enc).logits.float(), dim=-1).cpu().numpy())
        probs = np.concatenate(probs)
        pred = test_df.copy()
        pred["pred_sentiment_label"] = values[np.argmax(probs, axis=1)].astype(int)
        pred["pred_expected_score"] = np.clip(probs @ values, min(SENTIMENT_LABELS), max(SENTIMENT_LABELS))
        result = evaluate_predictions(pred, threshold=float("nan"), include_stress=False)
        result["meta"] = {"status": "ok", "model": baseline_cfg["name"], "label_values": baseline_cfg["label_values"],
                          "n_test": len(pred), "evaluated_at": utc_now_iso(), "stress_note": BASELINE_STRESS_NOTE}
        return result
    except Exception as exc:  # noqa: BLE001
        print(f"⚠ 零样本基线加载或运行失败，已跳过：{type(exc).__name__}: {exc}")
        return {"meta": {"status": "skipped", "model": baseline_cfg["name"], "error": f"{type(exc).__name__}: {exc}",
                         "evaluated_at": utc_now_iso()}}


# 零样本基线与测试评估受同一个“只运行一次”保护；两者都完成后才写入测试标记
if globals().get("RUN_TEST_EVALUATION", False):
    baseline_test = read_unified_csv(FILES["split_test"])
    baseline_test = baseline_test[baseline_test["lang"] == LANG].reset_index(drop=True)
    assert (baseline_test["label_method"] == "direct").all(), "测试集中出现非 direct 样本！"
    baseline_results = run_zero_shot_baseline(baseline_test, BASELINE_MODELS[LANG])
    update_lang_entry(FILES["baseline_results"], LANG, baseline_results)
    if baseline_results["meta"]["status"] == "ok":
        print("基线情感倾向：", json.dumps(to_jsonable({k: v for k, v in baseline_results["overall"]["sentiment"].items()
                                                    if k != "confusion_matrix"}), ensure_ascii=False))
    print(f"基线结果 → {FILES['baseline_results']}")
    save_json({"lang": LANG, "evaluated_at": utc_now_iso(), "model_sha256": TEST_MODEL_SHA256}, FILES["test_marker"])
    RUN_TEST_EVALUATION = False
    TEST_EVALUATED_NOW = True
    print(f"测试标记已写入 → {FILES['test_marker']}")
else:
    print("⏭ 跳过零样本基线：本次未运行测试评估（测试集只评估一次）。")

## 8. 阶段7：推理接口与示例

- `analyze(text: str) -> dict` 与 `analyze_batch(texts: list[str]) -> list[dict]`
- 返回 `sentiment`（`label` 为 −2..2 整数、`expected_score` ∈ [−2, 2]、各类 `probabilities`、`confidence`）、`stress`（`score`、`label`、`confidence`）、按 `LANG` 的 `disclaimer`；仅当输入包含明确的自伤 / 自杀相关关键词时附加 `help_message`。
- 空输入或少于 5 个词（en）/ 5 个字（zh）时，`confidence = "insufficient_info"`。
- **help_message 只是基于简单关键词规则的粗略提示，不是风险评估。**
- 每次调用把输入文本的 SHA-256 哈希与输出追加到 `reports/inference_log.jsonl`，不保存原文。

In [ ]:
# 阶段7：推理接口
INFERENCE_BUNDLE = {}


def _file_state(path):
    """返回文件的 (mtime_ns, 字节数)；不存在时返回 None。"""
    if not os.path.exists(path):
        return None
    st = os.stat(path)
    return st.st_mtime_ns, st.st_size


def inference_cache_key():
    """推理缓存键：LANG + model.pt 的文件状态 + 两个校准文件的 mtime；任一变化都会触发重新加载与指纹核对。"""
    return (LANG, _file_state(FILES["model_weights"]), _file_state(FILES["temperature"]),
            _file_state(FILES["thresholds"]))


def get_inference_bundle():
    """惰性加载当前 LANG 的模型、分词器、温度与压力阈值；模型或校准文件变化时自动重新加载，并核对模型指纹。"""
    key = inference_cache_key()
    if INFERENCE_BUNDLE.get("key") != key:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model, tokenizer, _ = load_trained_model(device)
        sha = model_fingerprint()
        t_sent, t_stress, threshold = load_calibration(LANG, sha)  # 指纹不一致时报错，提示重新运行阶段5
        INFERENCE_BUNDLE.clear()
        INFERENCE_BUNDLE.update({"key": key, "lang": LANG, "model_sha256": sha, "device": device, "model": model,
                                 "tokenizer": tokenizer, "t_sent": t_sent, "t_stress": t_stress,
                                 "threshold": threshold})
    return INFERENCE_BUNDLE


def count_input_units(text, lang):
    """统计输入长度：en 为空白分隔的词数；zh 为汉字数 + 字母数字词数。"""
    if lang == "zh":
        return len(re.findall(CJK_CHAR_PATTERN, text)) + len(re.findall(ALNUM_TOKEN_PATTERN, text))
    return len(text.split())


def contains_help_keyword(text):
    """简单关键词规则（casefold 子串匹配，检查所有语言的关键词表）：是否包含明确的自伤 / 自杀相关关键词。仅为粗略提示，不是风险评估。"""
    folded = text.casefold()
    return any(kw.casefold() in folded for keywords in SELF_HARM_KEYWORDS.values() for kw in keywords)


def sentiment_confidence(max_prob):
    """按最大类别概率划分情感置信度：high / medium / low。"""
    if max_prob >= SENTIMENT_CONFIDENCE_BANDS["high"]:
        return "high"
    if max_prob >= SENTIMENT_CONFIDENCE_BANDS["medium"]:
        return "medium"
    return "low"


def stress_confidence(score, threshold):
    """按 |score − threshold| 划分压力置信度：high / medium / low。"""
    margin = abs(score - threshold)
    if margin >= STRESS_CONFIDENCE_BANDS["high"]:
        return "high"
    if margin >= STRESS_CONFIDENCE_BANDS["medium"]:
        return "medium"
    return "low"


def insufficient_info_result():
    """输入信息不足时的固定输出（均匀概率、confidence = insufficient_info）。"""
    uniform = round(1.0 / len(SENTIMENT_LABELS), OUTPUT_DECIMALS)
    return {
        "sentiment": {"label": 0, "expected_score": 0.0,
                      "probabilities": {str(v): uniform for v in SENTIMENT_LABELS},
                      "confidence": "insufficient_info"},
        "stress": {"score": 0.0, "label": False, "confidence": "insufficient_info"},
    }


def log_inference(text, output):
    """把输入文本的 SHA-256 哈希与输出追加写入 reports/inference_log.jsonl（不保存原文）。"""
    if not INFERENCE_LOG_ENABLED:
        return
    record = {"ts": utc_now_iso(), "sha256": hashlib.sha256(text.encode("utf-8")).hexdigest(),
              "lang": LANG, "output": output}
    assert_not_raw(FILES["inference_log"])
    with open(FILES["inference_log"], "a", encoding="utf-8") as f:
        f.write(json.dumps(to_jsonable(record), ensure_ascii=False) + "\n")


def analyze_batch(texts):
    """批量分析文本的情感倾向与压力表达倾向，返回与输入等长的结果列表（结构同 analyze）。"""
    if not isinstance(texts, (list, tuple)):
        raise TypeError("texts 必须是字符串列表")
    clean = []
    for t in texts:
        if t is None:
            t = ""
        if not isinstance(t, str):
            raise TypeError(f"输入必须是字符串，收到 {type(t).__name__}")
        clean.append(t)
    results = [None] * len(clean)
    # 关键步骤：空输入或过短输入直接返回 insufficient_info，不调用模型
    valid_idx = [i for i, t in enumerate(clean)
                 if t.strip() and count_input_units(t, LANG) >= MIN_INPUT_UNITS[LANG]]
    valid_set = set(valid_idx)
    for i in range(len(clean)):
        if i not in valid_set:
            results[i] = insufficient_info_result()
    if valid_idx:
        b = get_inference_bundle()
        frame = pd.DataFrame({"text": [clean[i] for i in valid_idx], "sentiment": np.nan, "stress": np.nan})
        raw = predict_raw(b["model"], make_loader(frame, b["tokenizer"], EVAL_BATCH_SIZE, False, SEED), b["device"])
        post = postprocess(raw, b["t_sent"], b["t_stress"])
        for j, i in enumerate(valid_idx):
            probs = post["probs"][j]
            score = float(post["stress_score"][j])
            results[i] = {
                "sentiment": {
                    "label": int(post["label"][j]),
                    "expected_score": float(np.clip(round(float(post["expected"][j]), OUTPUT_DECIMALS),
                                                    min(SENTIMENT_LABELS), max(SENTIMENT_LABELS))),
                    "probabilities": {str(v): round(float(p), OUTPUT_DECIMALS) for v, p in zip(SENTIMENT_LABELS, probs)},
                    "confidence": sentiment_confidence(float(probs.max())),
                },
                "stress": {"score": round(score, OUTPUT_DECIMALS), "label": bool(score >= b["threshold"]),
                           "confidence": stress_confidence(score, b["threshold"])},
            }
    for i, t in enumerate(clean):
        results[i]["disclaimer"] = DISCLAIMERS[LANG]
        if contains_help_keyword(t):
            results[i]["help_message"] = HELP_MESSAGES[LANG]
        log_inference(t, results[i])
    return results


def analyze(text):
    """分析单条文本的情感倾向与压力表达倾向（仅为文本倾向估计），返回结果字典。"""
    return analyze_batch([text])[0]


def validate_output_schema(result):
    """检查 analyze() 输出是否符合接口约定（字段、类型与取值范围），不符合时抛出 AssertionError。"""
    s, st = result["sentiment"], result["stress"]
    assert isinstance(s["label"], int) and not isinstance(s["label"], bool) and s["label"] in SENTIMENT_LABELS
    assert isinstance(s["expected_score"], float)
    assert min(SENTIMENT_LABELS) <= s["expected_score"] <= max(SENTIMENT_LABELS)
    assert set(s["probabilities"]) == {str(v) for v in SENTIMENT_LABELS}
    assert abs(sum(s["probabilities"].values()) - 1.0) < 1e-2
    assert s["confidence"] in ("high", "medium", "low", "insufficient_info")
    assert isinstance(st["score"], float) and 0.0 <= st["score"] <= 1.0
    assert isinstance(st["label"], bool)
    assert st["confidence"] in ("high", "medium", "low", "insufficient_info")
    assert result["disclaimer"] == DISCLAIMERS[LANG]
    assert set(result) <= {"sentiment", "stress", "disclaimer", "help_message"}
    if "help_message" in result:
        assert result["help_message"] == HELP_MESSAGES[LANG]
    return True

In [ ]:
# 阶段7：示例（示例文本为虚构，仅用于演示接口）
EXAMPLE_TEXTS = {
    "en": [
        "I finally finished my project today and everyone loved the presentation.",
        "Deadlines keep piling up and I cannot sleep because of all the pressure at work.",
        "Too short.",
        "",
    ],
    "zh": [
        "今天终于完成了项目，大家都很喜欢我的展示。",
        "最近工作压力很大，截止日期一个接一个，晚上总是睡不着。",
        "还行",
        "",
    ],
}
example_results = analyze_batch(EXAMPLE_TEXTS[LANG])
for text, res in zip(EXAMPLE_TEXTS[LANG], example_results):
    validate_output_schema(res)
    print(f"输入：{text!r}")
    print(json.dumps(res, ensure_ascii=False, indent=2))
single = analyze(EXAMPLE_TEXTS[LANG][0])
validate_output_schema(single)
print(f"\n推理记录（只含哈希）→ {FILES['inference_log']}")

## 9. 每个阶段的输出文件列表

In [ ]:
# 按阶段列出输出文件是否存在及其大小
STAGE_OUTPUTS = {
    "2. 配置区": [FILES["label_mapping"]],
    "阶段2 标签映射": [FILES["unified"], FILES["mapping_distribution"], FILES["mapping_notes"], FILES["data_inventory"]],
    "阶段3 划分与泄露检查": [FILES["split_train"], FILES["split_val"], FILES["split_test"], FILES["leakage_check"]],
    "阶段4 训练": [FILES["model_weights"], FILES["tokenizer_dir"], FILES["encoder_config_dir"], FILES["train_config"],
                FILES["experiment_log"]],
    "阶段5 校准与阈值": [FILES["temperature"], FILES["thresholds"]],
    "阶段6 测试评估": [FILES["test_results"], FILES["error_cases"], FILES["baseline_results"], FILES["test_marker"]],
    "阶段7 推理": [FILES["inference_log"]],
}


def path_size(path):
    """返回文件或目录的总字节数；不存在时返回 None。"""
    if os.path.isfile(path):
        return os.path.getsize(path)
    if os.path.isdir(path):
        return sum(os.path.getsize(os.path.join(root, f)) for root, _, fs in os.walk(path) for f in fs)
    return None


def format_size(n):
    """把字节数格式化为易读字符串。"""
    for unit in ("B", "KB", "MB", "GB"):
        if n < 1024:
            return f"{n:.0f} {unit}" if unit == "B" else f"{n:.1f} {unit}"
        n /= 1024
    return f"{n:.1f} TB"


def list_stage_outputs(stage_outputs):
    """按阶段打印输出文件是否存在及大小，返回缺失文件列表。"""
    missing = []
    for stage, paths in stage_outputs.items():
        print(f"\n[{stage}]")
        for p in paths:
            size = path_size(p)
            rel = os.path.relpath(p, DRIVE_ROOT)
            if size is None:
                missing.append(p)
                print(f"  ✗ {rel}（不存在）")
            else:
                print(f"  ✓ {rel}  {format_size(size)}")
    return missing


MISSING_OUTPUTS = list_stage_outputs(STAGE_OUTPUTS)
print(f"\n缺失文件数：{len(MISSING_OUTPUTS)}")

## 附：表述自检

检查本 notebook 源码中不出现配置区 `BANNED_WORDING` 列出的禁用表述（免责声明原文除外）。输出措辞只使用“情感倾向”“压力表达倾向”“可能相关”等。

In [ ]:
# 表述自检：扫描 notebook 源码中的禁用表述
def get_notebook_sources():
    """获取本 notebook 全部单元源码：优先 Colab 接口，其次 SELF_CHECK_NOTEBOOK_PATH，最后 IPython 执行历史。"""
    try:
        from google.colab import _message
        nb = _message.blocking_request("get_ipynb", timeout_sec=30)["ipynb"]
        return ["".join(c["source"]) if isinstance(c["source"], list) else c["source"] for c in nb["cells"]]
    except Exception:  # noqa: BLE001
        pass
    if SELF_CHECK_NOTEBOOK_PATH and os.path.exists(SELF_CHECK_NOTEBOOK_PATH):
        nb = load_json(SELF_CHECK_NOTEBOOK_PATH)
        return ["".join(c["source"]) if isinstance(c["source"], list) else c["source"] for c in nb["cells"]]
    try:
        return [s for s in get_ipython().user_ns.get("In", []) if s]  # noqa: F821
    except NameError:
        return []


def find_wording_violations(sources):
    """返回违规行列表：先移除白名单短语（免责声明等），跳过带允许标记的行，再匹配禁用词（casefold）。"""
    violations = []
    banned = [w.casefold() for w in BANNED_WORDING]
    for ci, src in enumerate(sources):
        cleaned = src
        for phrase in WORDING_WHITELIST:
            cleaned = cleaned.replace(phrase, "")
        for li, line in enumerate(cleaned.splitlines(), start=1):
            if WORDING_ALLOW_MARKER in line:
                continue
            hits = [w for w in banned if w in line.casefold()]
            if hits:
                violations.append((ci, li, hits, line.strip()[:120]))
    return violations


notebook_sources = get_notebook_sources()
if not notebook_sources:
    print("⚠ 无法获取 notebook 源码，跳过表述自检（可设置 SELF_CHECK_NOTEBOOK_PATH）。")
else:
    wording_violations = find_wording_violations(notebook_sources)
    for v in wording_violations:
        print("违规：", v)
    assert not wording_violations, "notebook 中出现禁用表述，请修改措辞"
    print(f"表述自检通过：扫描 {len(notebook_sources)} 个单元，未发现禁用表述。")